In [ ]:
# +-- Cell -1: scheduled-run identity -----------------------------------------
# MUST execute before Cell 0.  Cell 1 reads RUN_ID as
#     RUN_ID = os.getenv('PS2_RUN_ID', str(uuid.uuid4()))
# so the value has to exist in the environment by the time that cell runs.
#
# WHY A PLAIN uuid4 IS NOT SAFE FOR A DAILY SCHEDULE
# The RDS loader selects a partition with
#     latest_partition() -> max over the tuple (computed_date, run_id)
# When several runs share one computed_date -- which is exactly what a daily
# schedule against a fixed 24-month dump produces -- the tie is broken by the
# ORDER of run_id.  With a random uuid4 that order is arbitrary, so the loader
# can serve a three-week-old run and still report success.
#
# THE ID MUST SATISFY THREE CONSTRAINTS AT ONCE
#   1. loader regex  run_id=([0-9a-f\-]+)/   -> hex and dashes only.  Any other
#      character makes the path unmatchable and the table is silently skipped.
#   2. chronological sort                     -> so "latest" means latest.
#   3. uuid.UUID(name) must parse             -> Cell 1's stale-scratch cleaner
#      does uuid.UUID(dir.name) and skips on ValueError.  A non-UUID id means
#      the daily Spark scratch directories are never cleaned, and with
#      MIN_SPARK_FREE_GIB = 20 the job hard-fails a few weeks in.
#
# Hex epoch-seconds in the first 8 characters satisfies all three.  Valid until
# the year 2106.
#
# Idempotent: an id supplied by the scheduling job always wins.
import os, uuid, datetime as dt

if not os.environ.get('PS2_RUN_ID'):
    _ts = dt.datetime.now(dt.timezone.utc)
    os.environ['PS2_RUN_ID'] = str(
        uuid.UUID(format(int(_ts.timestamp()), '08x') + uuid.uuid4().hex[8:])
    )

print('PS2_RUN_ID   =', os.environ['PS2_RUN_ID'])
print('PS2_RUN_MODE =', os.environ.get('PS2_RUN_MODE', '(unset -> REPLAY)'))

# PS2 v2.5.1 — Governed failure-pattern intelligence with PS1-compatible Silver labels

This scheduled SageMaker notebook separates two truths that serve different purposes:

1. **PS1-compatible future label** — rebuilt directly from the stable S3 Silver Parquet
   exports. `will_hardware_oos_3d = 1` for a device-day when a hardware-OOS `Set` event
   occurs on calendar day +1, +2, or +3. This is the exact contract shared by the
   attached TVM and Gate PS1 notebooks and is parameterized for Validator as well.
2. **Governed PS2 episode label** — a timestamped, uncommanded, non-maintenance hardware
   OOS onset linked to an outage episode. This remains the causal/operational truth for
   precursor, serial, repair, cluster, exposure, and drift analytics.

The notebook never silently substitutes raw OOS state, `will_fail_3d`, chargeability, or
same-day failure evidence for either definition. It publishes their alignment explicitly.

Run **Cell 0 onward / Run All**. Scheduled production runs should set
`PS2_RUN_MODE=PRODUCTION`; historical work should use `REPLAY` or `DIAGNOSTIC` so the
dispatcher prefix remains protected.


### v2.5.1 execution-safety correction

Spark scratch is selected from the writable SageMaker volume with the most free space,
checked before Spark starts, and isolated by run ID. Dashboard tables are collected once;
row limits and RDS key checks run on that same bounded materialization rather than launching
three copies of an expensive lineage. The timestamp-edge window is reduced to one compact,
reusable aggregate before precursor, topology, lead/lag, and drift outputs are derived.


## What v2.5.1 includes

- Stable, case-insensitive direct readers for
  `chicago/silver/device_event_enriched/` and `chicago/silver/dim_device/`; no Unity
  Storage table UUID is needed for the PS1-compatible label.
- Exact three-calendar-day PS1 label parity, with right-censoring protection based on the
  source as-of timestamp rather than the notebook run date.
- Separate label prevalence, horizon, parity, definition-alignment, and optional PS1 model
  performance tables (precision, recall, F1, specificity, balanced accuracy, Brier score,
  ROC AUC, and PR AUC).
- Canonical category mapping and explicit scoping to TVM, Gate, and Validator, so valid
  out-of-scope devices no longer fail the category-quality gate.
- Validated failures are allocated to the nearest governed OOS onset on the device/day;
  they are no longer copied onto every event from that day.
- Component-serial coverage is measured on the governed OOS-onset population.
- A blocked production run completes its audit outputs first and only then returns a
  failing scheduler status when fail-fast is enabled.


- Storage-aware Spark scratch selection and an early free-space gate.
- One Spark action per dashboard table, followed by in-memory key validation.
- Explicit cache lifecycle management and a run-completion marker, so a partially
  published run is distinguishable from a completed run.


In [ ]:
# ── Cell 0: SageMaker runtime bootstrap and compatibility gate ────────────────
import os, sys, re, json, shutil, subprocess, importlib, importlib.util
from importlib import metadata as importlib_metadata
from pathlib import Path


def env_flag(name, default=False):
    value = os.getenv(name, str(default)).strip().lower()
    if value not in {'true', 'false'}:
        raise ValueError(f'{name} must be true or false; observed {value!r}.')
    return value == 'true'


python_minor = sys.version_info[:2]
if (3, 8) <= python_minor <= (3, 11):
    # Delta 3.3.2 is built on Spark 3.5.3 and is the conservative production profile.
    RUNTIME_PROFILE = {
        'name': 'spark35_delta33', 'pyspark': '3.5.3', 'delta_spark': '3.3.2',
        'python': f'{python_minor[0]}.{python_minor[1]}', 'minimum_java': 11,
    }
elif python_minor == (3, 12):
    # Spark 4 is the supported path for Python 3.12; Delta 4.0.1 matches Spark 4.0.1.
    RUNTIME_PROFILE = {
        'name': 'spark40_delta40', 'pyspark': '4.0.1', 'delta_spark': '4.0.1',
        'python': '3.12', 'minimum_java': 17,
    }
else:
    raise RuntimeError(
        f'Unsupported Python {sys.version.split()[0]}. Use a SageMaker Python 3.11 or 3.12 kernel.'
    )

AUTO_INSTALL_RUNTIME = env_flag('PS2_AUTO_INSTALL_RUNTIME', True)
AUTO_INSTALL_JAVA = env_flag('PS2_AUTO_INSTALL_JAVA', AUTO_INSTALL_RUNTIME)


def distribution_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


required_runtime = {
    'pyspark': RUNTIME_PROFILE['pyspark'],
    'delta-spark': RUNTIME_PROFILE['delta_spark'],
}
runtime_changes = {
    name: {'installed': distribution_version(name), 'required': required}
    for name, required in required_runtime.items()
    if distribution_version(name) != required
}

reporting_requirements = {
    'boto3': 'boto3>=1.34,<2',
    'pandas': 'pandas>=2.0,<3',
    'pyarrow': 'pyarrow>=11,<22',
    'matplotlib': 'matplotlib>=3.7,<4',
    'seaborn': 'seaborn>=0.13,<0.14',
}
missing_reporting = [
    requirement for module, requirement in reporting_requirements.items()
    if importlib.util.find_spec(module) is None
]

install_requirements = [
    f'{name}=={required}' for name, required in required_runtime.items()
    if name in runtime_changes
] + missing_reporting

if install_requirements:
    already_imported = sorted(
        name for name in ('pyspark', 'delta', 'py4j') if name in sys.modules
    )
    if already_imported:
        raise RuntimeError(
            'Spark/Delta packages need correction but related modules are already loaded: '
            f'{already_imported}. Restart the kernel, then use Run All from Cell 0.'
        )
    if not AUTO_INSTALL_RUNTIME:
        raise RuntimeError(
            'Required runtime packages are absent or incompatible: '
            f'{runtime_changes}; missing reporting packages={missing_reporting}. '
            'Install them in the SageMaker image/lifecycle configuration, or set '
            'PS2_AUTO_INSTALL_RUNTIME=true for an interactive bootstrap.'
        )
    print('Installing pinned PS2 runtime packages:', install_requirements)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
        '--no-input', *install_requirements,
    ])
    importlib.invalidate_caches()

post_install_versions = {name: distribution_version(name) for name in required_runtime}
bad_versions = {
    name: {'installed': post_install_versions[name], 'required': required}
    for name, required in required_runtime.items()
    if post_install_versions[name] != required
}
if bad_versions:
    raise RuntimeError(
        f'Runtime installation did not produce the required versions: {bad_versions}. '
        'Restart the kernel and rerun Cell 0; do not continue to the publish cell.'
    )

def locate_java():
    candidates = []
    for prefix in (os.getenv('CONDA_PREFIX'), sys.prefix):
        if prefix:
            candidates.extend([
                Path(prefix) / 'bin' / 'java',
                Path(prefix) / 'lib' / 'jvm' / 'bin' / 'java',
            ])
    if os.getenv('JAVA_HOME'):
        candidates.append(Path(os.environ['JAVA_HOME']) / 'bin' / 'java')
    path_java = shutil.which('java')
    if path_java:
        candidates.append(Path(path_java))
    for root in (Path('/usr/lib/jvm'), Path('/opt/java')):
        if root.exists():
            candidates.extend(sorted(root.glob('*/bin/java')))

    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        key = str(candidate)
        if key not in seen and candidate.is_file() and os.access(candidate, os.X_OK):
            return key
        seen.add(key)
    return None


def inspect_java(java_path):
    if not java_path:
        return None, None
    check = subprocess.run(
        [java_path, '-version'], capture_output=True, text=True, check=False
    )
    lines = (check.stderr or check.stdout).splitlines()
    version_line = lines[0].strip() if lines else ''
    match = re.search(r'version\s+"(?:(1)\.)?(\d+)', version_line)
    return version_line, int(match.group(2)) if match else None


def install_openjdk17():
    manager = (
        os.getenv('CONDA_EXE') or shutil.which('mamba') or
        shutil.which('conda') or shutil.which('micromamba')
    )
    if not manager:
        raise RuntimeError(
            'Java 17 is missing and no Conda/Mamba executable is available for the automatic '
            'installation. Use a SageMaker image containing OpenJDK 17.'
        )
    command = [
        manager, 'install', '--yes', '--prefix', sys.prefix,
        '--channel', 'conda-forge', '--freeze-installed', 'openjdk=17',
    ]
    print(f'Installing pinned Java runtime openjdk=17 into {sys.prefix}')
    try:
        subprocess.check_call(command)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Automatic OpenJDK 17 installation failed. Confirm that this SageMaker kernel can '
            'write to its Conda environment and reach conda-forge, or use a custom image with '
            'OpenJDK 17 preinstalled.'
        ) from exc
    os.environ['PATH'] = f'{Path(sys.prefix) / "bin"}{os.pathsep}{os.environ.get("PATH", "")}'


java_bin = locate_java()
JAVA_VERSION_LINE, java_major = inspect_java(java_bin)
if java_bin is None or java_major is None or java_major < RUNTIME_PROFILE['minimum_java']:
    if not AUTO_INSTALL_JAVA:
        raise RuntimeError(
            f'Java >= {RUNTIME_PROFILE["minimum_java"]} is required; observed '
            f'{JAVA_VERSION_LINE or "no Java binary"}. Set PS2_AUTO_INSTALL_JAVA=true or '
            'use a SageMaker image containing OpenJDK 17.'
        )
    install_openjdk17()
    java_bin = locate_java()
    JAVA_VERSION_LINE, java_major = inspect_java(java_bin)

if java_bin is None or java_major is None or java_major < RUNTIME_PROFILE['minimum_java']:
    raise RuntimeError(
        f'OpenJDK installation completed but a compatible Java executable was not found. '
        f'Observed: {JAVA_VERSION_LINE or "no Java binary"}; required Java >= '
        f'{RUNTIME_PROFILE["minimum_java"]}. Restart the kernel and rerun Cell 0.'
    )

os.environ['JAVA_HOME'] = str(Path(java_bin).resolve().parent.parent)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


def detected_memory_bytes():
    '''Return the effective container/host memory limit, preferring the lowest real limit.'''
    candidates = []
    for path in (
        Path('/sys/fs/cgroup/memory.max'),
        Path('/sys/fs/cgroup/memory/memory.limit_in_bytes'),
    ):
        try:
            value = path.read_text(encoding='utf-8').strip()
            if value.isdigit():
                parsed = int(value)
                if 512 * 1024**2 <= parsed < 2**60:
                    candidates.append(parsed)
        except OSError:
            pass
    try:
        meminfo = Path('/proc/meminfo').read_text(encoding='utf-8')
        match = re.search(r'^MemTotal:\s+(\d+)\s+kB', meminfo, flags=re.MULTILINE)
        if match:
            candidates.append(int(match.group(1)) * 1024)
    except OSError:
        pass
    return min(candidates) if candidates else 16 * 1024**3


EFFECTIVE_MEMORY_BYTES = detected_memory_bytes()
EFFECTIVE_MEMORY_GIB = EFFECTIVE_MEMORY_BYTES / 1024**3
if EFFECTIVE_MEMORY_GIB < 8:
    raise RuntimeError(
        f'This kernel exposes only {EFFECTIVE_MEMORY_GIB:.1f} GiB. PS2 local Spark requires '
        'at least 8 GiB for a guarded test and substantially more for the 210-day production window.'
    )

default_driver_gib = max(
    4,
    min(int(EFFECTIVE_MEMORY_GIB * 0.60), max(4, int(EFFECTIVE_MEMORY_GIB) - 3)),
)
SPARK_DRIVER_MEMORY = os.getenv('PS2_SPARK_DRIVER_MEMORY', f'{default_driver_gib}g').strip().lower()
if not re.fullmatch(r'\d+[gm]', SPARK_DRIVER_MEMORY):
    raise ValueError(
        f'PS2_SPARK_DRIVER_MEMORY must use a whole-number g/m suffix, e.g. 24g; '
        f'observed {SPARK_DRIVER_MEMORY!r}.'
    )
driver_gib_for_sizing = (
    int(SPARK_DRIVER_MEMORY[:-1]) if SPARK_DRIVER_MEMORY.endswith('g')
    else max(1, int(SPARK_DRIVER_MEMORY[:-1]) // 1024)
)
SPARK_DRIVER_MAX_RESULT_SIZE = os.getenv(
    'PS2_SPARK_DRIVER_MAX_RESULT_SIZE',
    f'{max(1, min(4, driver_gib_for_sizing // 4))}g',
).strip().lower()
if not re.fullmatch(r'\d+[gm]', SPARK_DRIVER_MAX_RESULT_SIZE):
    raise ValueError('PS2_SPARK_DRIVER_MAX_RESULT_SIZE must use a whole-number g/m suffix.')
default_local_cores = max(
    2,
    min(8, os.cpu_count() or 2, max(2, driver_gib_for_sizing // 2)),
)
SPARK_LOCAL_CORES = int(os.getenv('PS2_SPARK_LOCAL_CORES', str(default_local_cores)))
if SPARK_LOCAL_CORES < 1:
    raise ValueError('PS2_SPARK_LOCAL_CORES must be >= 1.')
SPARK_MASTER = os.getenv('PS2_SPARK_MASTER', f'local[{SPARK_LOCAL_CORES}]')
SPARK_SHUFFLE_PARTITIONS = int(os.getenv(
    'PS2_SHUFFLE_PARTITIONS', str(max(128, SPARK_LOCAL_CORES * 32))
))

# Driver heap is a deploy-time setting. It must be supplied before PySpark creates its
# Java gateway; setting spark.driver.memory later on SparkSession.builder is insufficient.
loaded_pyspark = sys.modules.get('pyspark')
loaded_spark_context = getattr(loaded_pyspark, 'SparkContext', None)
if loaded_spark_context is not None and getattr(loaded_spark_context, '_gateway', None) is not None:
    raise RuntimeError(
        'A Spark JVM has already been launched in this kernel, so Cell 0 cannot apply the '
        f'new {SPARK_DRIVER_MEMORY} driver heap. Restart the kernel and use Run All from Cell 0.'
    )

submit_override = os.getenv('PS2_PYSPARK_SUBMIT_ARGS', '').strip()
if submit_override:
    os.environ['PYSPARK_SUBMIT_ARGS'] = (
        submit_override if submit_override.endswith('pyspark-shell')
        else f'{submit_override} pyspark-shell'
    )
else:
    os.environ['PYSPARK_SUBMIT_ARGS'] = f'--driver-memory {SPARK_DRIVER_MEMORY} pyspark-shell'

print(json.dumps({
    'runtime_profile': RUNTIME_PROFILE,
    'java': JAVA_VERSION_LINE,
    'java_home': os.environ['JAVA_HOME'],
    'auto_install_runtime': AUTO_INSTALL_RUNTIME,
    'auto_install_java': AUTO_INSTALL_JAVA,
    'effective_memory_gib': round(EFFECTIVE_MEMORY_GIB, 1),
    'spark_driver_memory': SPARK_DRIVER_MEMORY,
    'spark_driver_max_result_size': SPARK_DRIVER_MAX_RESULT_SIZE,
    'spark_master': SPARK_MASTER,
    'spark_shuffle_partitions': SPARK_SHUFFLE_PARTITIONS,
}, indent=2))



In [ ]:
# ── Cell 1: configuration ────────────────────────────────────────────────────
import os, json, uuid, hashlib, shutil, textwrap
import datetime as dt
from pathlib import Path

NOTEBOOK_VERSION = 'ps2_failure_patterns_v2_5_4_union_minutes'
CITY_ID = os.getenv('PS2_CITY_ID', 'CHI')
RUN_ID = os.getenv('PS2_RUN_ID', str(uuid.uuid4()))
RUN_TS = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
# Cell 8 sets this to PASS or BLOCKED after the governed OOS/data-quality gate.
# Do not infer a PASS default: publishing without that gate would be unsafe.
QUALITY_STATUS = None

# The attached Chicago catalogue is a historical snapshot, so an interactive Run All
# defaults to an isolated REPLAY. Scheduled jobs must explicitly set
# PS2_RUN_MODE=PRODUCTION after the upstream batch is current.
legacy_production_mode = os.getenv('PS2_PRODUCTION_MODE')
if legacy_production_mode is None:
    default_run_mode = 'REPLAY'
elif legacy_production_mode.strip().lower() in {'true', 'false'}:
    default_run_mode = 'PRODUCTION' if legacy_production_mode.strip().lower() == 'true' else 'REPLAY'
else:
    raise ValueError('PS2_PRODUCTION_MODE must be true or false when supplied.')
RUN_MODE = os.getenv('PS2_RUN_MODE', default_run_mode).strip().upper()
if RUN_MODE not in {'PRODUCTION', 'REPLAY', 'DIAGNOSTIC'}:
    raise ValueError('PS2_RUN_MODE must be PRODUCTION, REPLAY, or DIAGNOSTIC.')
PRODUCTION_MODE = RUN_MODE == 'PRODUCTION'
REPLAY_MODE = RUN_MODE == 'REPLAY'
DIAGNOSTIC_MODE = RUN_MODE == 'DIAGNOSTIC'
AS_OF_DATE_OVERRIDE = os.getenv('PS2_AS_OF_DATE', '').strip()       # YYYY-MM-DD, optional
ANALYSIS_LOOKBACK_DAYS = int(os.getenv('PS2_ANALYSIS_LOOKBACK_DAYS', '210'))
RECENT_WINDOW_DAYS = int(os.getenv('PS2_RECENT_WINDOW_DAYS', '28'))
CASCADE_GAP_MINUTES = int(os.getenv('PS2_CASCADE_GAP_MINUTES', '30'))
PRECURSOR_LOOKBACK_HOURS = int(os.getenv('PS2_PRECURSOR_LOOKBACK_HOURS', '48'))
MIN_PATTERN_SUPPORT = int(os.getenv('PS2_MIN_PATTERN_SUPPORT', '50'))
MIN_COMPONENT_SERIAL_COVERAGE = float(os.getenv('PS2_MIN_COMPONENT_SERIAL_COVERAGE', '0.40'))
MIN_CATEGORY_MAPPING_COVERAGE = float(os.getenv('PS2_MIN_CATEGORY_MAPPING_COVERAGE', '0.98'))
MAX_FAILURE_LINK_HOURS = int(os.getenv('PS2_MAX_FAILURE_LINK_HOURS', '24'))

# Exact PS1 target contract learned from the attached TVM/Gate notebooks.  The same
# category-parameterized construction applies to VALIDATOR.
TARGET_DEVICE_CATEGORIES = ('TVM', 'GATE', 'VALIDATOR')
DEVICE_CATEGORY_ALIASES = {
    'TVM': 'TVM', 'TICKET VENDING MACHINE': 'TVM', 'TICKET_VENDING_MACHINE': 'TVM',
    'GATE': 'GATE', 'GATES': 'GATE', 'FAREGATE': 'GATE', 'FARE GATE': 'GATE',
    'VALIDATOR': 'VALIDATOR', 'VALIDATORS': 'VALIDATOR', 'BUS VALIDATOR': 'VALIDATOR',
    'BUS_VALIDATOR': 'VALIDATOR',
}
PS1_LABEL_COL = os.getenv('PS2_PS1_LABEL_COL', 'will_hardware_oos_3d')
PS1_LEGACY_SLA_LABEL_COL = os.getenv('PS2_PS1_LEGACY_SLA_LABEL_COL', 'will_fail_3d')
LABEL_HORIZON_DAYS = int(os.getenv('PS2_LABEL_HORIZON_DAYS', '3'))
LABEL_DATA_LAG_DAYS = int(os.getenv('PS2_LABEL_DATA_LAG_DAYS', '0'))
if LABEL_HORIZON_DAYS < 1:
    raise ValueError('PS2_LABEL_HORIZON_DAYS must be >= 1.')
ENABLE_PS1_PREDICTION_EVAL = os.getenv('PS2_ENABLE_PS1_PREDICTION_EVAL', 'true').lower() == 'true'

MAX_SOURCE_LAG_DAYS = int(os.getenv('PS2_MAX_SOURCE_LAG_DAYS', '3'))
MAX_OUTPUT_ROWS = int(os.getenv('PS2_MAX_OUTPUT_ROWS', '500000'))
ENABLE_REPAIR_EFFECTIVENESS = os.getenv('PS2_ENABLE_REPAIR_EFFECTIVENESS', 'true').lower() == 'true'
fail_fast_raw = os.getenv(
    'PS2_FAIL_FAST_ON_CRITICAL_GATE', str(PRODUCTION_MODE)
).strip().lower()
if fail_fast_raw not in {'true', 'false'}:
    raise ValueError('PS2_FAIL_FAST_ON_CRITICAL_GATE must be true or false.')
FAIL_FAST_ON_CRITICAL_GATE = fail_fast_raw == 'true'

# The existing dispatcher watches the production prefix. Non-production modes are forced
# onto isolated prefixes so a historical replay cannot trigger the production RDS Loader.
ARTIFACT_BUCKET = os.getenv('PS2_ARTIFACT_BUCKET', 'cubic-mars-pm-s3-datalake-dev-artifacts-170202974600')
PRODUCTION_EXPORT_PREFIX = os.getenv('PS2_PRODUCTION_EXPORT_PREFIX', 'chicago/ps2_outputs').strip().strip('/')
default_export_prefix = (
    PRODUCTION_EXPORT_PREFIX if PRODUCTION_MODE else f'ps2_{RUN_MODE.lower()}_outputs'
)
EXPORT_PREFIX = os.getenv('PS2_EXPORT_PREFIX', default_export_prefix).strip().strip('/')
if not EXPORT_PREFIX:
    raise ValueError('PS2_EXPORT_PREFIX cannot be empty.')
if not PRODUCTION_MODE and (
    EXPORT_PREFIX == PRODUCTION_EXPORT_PREFIX or
    EXPORT_PREFIX.startswith(f'{PRODUCTION_EXPORT_PREFIX}/')
):
    raise RuntimeError(
        f'{RUN_MODE} mode cannot use the production dispatcher prefix '
        f'{PRODUCTION_EXPORT_PREFIX!r}. Use an isolated PS2_EXPORT_PREFIX.'
    )
OUTPUT_SCOPE = 'PRODUCTION_DISPATCH' if PRODUCTION_MODE else f'ISOLATED_{RUN_MODE}'

# Spark local mode writes cache blocks, shuffle spill, and sort files to spark.local.dir.
# SageMaker's /tmp commonly shares a small root volume, so select the writable candidate
# with the most capacity unless the scheduled job supplies an attached-volume override.
MIN_SPARK_FREE_GIB = float(os.getenv('PS2_MIN_SPARK_FREE_GIB', '20'))
MIN_PUBLISH_FREE_GIB = float(os.getenv('PS2_MIN_PUBLISH_FREE_GIB', '5'))
STALE_SPARK_DIR_HOURS = int(os.getenv('PS2_STALE_SPARK_DIR_HOURS', '24'))
CLEAN_STALE_SPARK_DIRS = os.getenv('PS2_CLEAN_STALE_SPARK_DIRS', 'true').strip().lower() == 'true'
if MIN_SPARK_FREE_GIB <= 0 or MIN_PUBLISH_FREE_GIB <= 0:
    raise ValueError('PS2 Spark free-space thresholds must be positive.')


def disk_free_gib(path):
    return shutil.disk_usage(str(path)).free / 1024**3


explicit_spark_local_dir = os.getenv('PS2_SPARK_LOCAL_DIR', '').strip()
scratch_candidates = []
if explicit_spark_local_dir:
    SPARK_LOCAL_DIR = Path(explicit_spark_local_dir).expanduser().resolve()
    SPARK_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    PS2_WORK_ROOT = SPARK_LOCAL_DIR.parent
    scratch_candidates.append((str(SPARK_LOCAL_DIR), disk_free_gib(SPARK_LOCAL_DIR)))
    auto_selected_scratch = False
else:
    # /home/sagemaker-user is normally the durable Studio volume; /opt/ml is useful in
    # Processing jobs; /tmp is retained only as a last candidate when it has real headroom.
    candidate_roots = [Path('/home/sagemaker-user'), Path('/opt/ml'), Path('/tmp')]
    usable_roots = []
    for candidate_root in candidate_roots:
        try:
            candidate_root.mkdir(parents=True, exist_ok=True)
            probe_root = candidate_root / 'ps2_runtime'
            probe_root.mkdir(parents=True, exist_ok=True)
            free_gib = disk_free_gib(probe_root)
            scratch_candidates.append((str(probe_root), free_gib))
            if os.access(probe_root, os.W_OK):
                usable_roots.append((free_gib, probe_root))
        except OSError as candidate_error:
            scratch_candidates.append((str(candidate_root), f'UNAVAILABLE: {candidate_error}'))
    if not usable_roots:
        raise RuntimeError(f'No writable Spark scratch candidate is available: {scratch_candidates}')
    _, PS2_WORK_ROOT = max(usable_roots, key=lambda item: item[0])
    spark_parent = PS2_WORK_ROOT / 'spark'
    spark_parent.mkdir(parents=True, exist_ok=True)
    SPARK_LOCAL_DIR = spark_parent / RUN_ID
    SPARK_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    auto_selected_scratch = True

SPARK_FREE_GIB_AT_START = disk_free_gib(SPARK_LOCAL_DIR)
if SPARK_FREE_GIB_AT_START < MIN_SPARK_FREE_GIB:
    raise RuntimeError(
        f'Insufficient Spark scratch space: {SPARK_FREE_GIB_AT_START:.1f} GiB free at '
        f'{SPARK_LOCAL_DIR}; required >= {MIN_SPARK_FREE_GIB:.1f} GiB. Attach or select a '
        'larger volume and set PS2_SPARK_LOCAL_DIR before Run All. '
        f'Candidates inspected: {scratch_candidates}'
    )

# Remove only old UUID-named PS2 scratch children beneath the auto-selected safe parent.
# Current/recent runs are never touched, and explicit user paths are never auto-cleaned.
if auto_selected_scratch and CLEAN_STALE_SPARK_DIRS:
    cutoff_ts = dt.datetime.now(dt.timezone.utc).timestamp() - STALE_SPARK_DIR_HOURS * 3600
    for stale_dir in SPARK_LOCAL_DIR.parent.iterdir():
        if not stale_dir.is_dir() or stale_dir.name == RUN_ID:
            continue
        try:
            uuid.UUID(stale_dir.name)
        except ValueError:
            continue
        try:
            if stale_dir.stat().st_mtime < cutoff_ts:
                shutil.rmtree(stale_dir)
        except OSError as cleanup_error:
            print(f'WARNING: could not clean stale PS2 Spark directory {stale_dir}: {cleanup_error}')

LOCAL_STAGE = Path(
    os.getenv('PS2_LOCAL_STAGE', str(PS2_WORK_ROOT / 'stage' / RUN_ID))
).expanduser().resolve()
LOCAL_STAGE.mkdir(parents=True, exist_ok=True)

# These are the confirmed locations in Chicago_catalog_snapshot dated 2026-07-31.
# Put environment overrides here when table storage is rebuilt; do not guess a path.
# The label readers intentionally use the stable Parquet exports used by PS1.
PS1_SILVER_BUCKET = os.getenv(
    'PS2_PS1_SILVER_BUCKET',
    'cubic-mars-pm-s3-datalake-dev-gold-170202974600',
)
PS1_SILVER_ROOT = os.getenv(
    'PS2_PS1_SILVER_ROOT',
    f's3://{PS1_SILVER_BUCKET}/chicago/silver',
).rstrip('/')
PS1_PREDICTIONS_ROOT = os.getenv(
    'PS2_PS1_PREDICTIONS_ROOT',
    f's3://{ARTIFACT_BUCKET}/chicago/device_ps1_cross_wired_daily',
).rstrip('/')

SOURCE = {
    'label_events': {
        'format': 'parquet',
        'path': os.getenv('PS2_LABEL_EVENTS_PATH', f'{PS1_SILVER_ROOT}/device_event_enriched/'),
        'logical_name': 'direct_s3_silver.device_event_enriched',
    },
    'label_dim_device': {
        'format': 'parquet',
        'path': os.getenv('PS2_LABEL_DIM_DEVICE_PATH', f'{PS1_SILVER_ROOT}/dim_device/'),
        'logical_name': 'direct_s3_silver.dim_device',
    },
    'events': {
        'format': 'delta',
        'path': os.getenv('PS2_EVENTS_PATH', 's3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/tables/2860d966-5040-44c1-bfe4-64b2975e703f'),
        'logical_name': 'mars_dev.silver.device_event_enriched',
    },
    'outage': {
        'format': 'delta',
        'path': os.getenv('PS2_OUTAGE_PATH', 's3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/tables/f4d06b77-5a4e-4d11-810e-9ba5422348ca'),
        'logical_name': 'mars_dev.silver.device_outage',
    },
    'failures': {
        'format': 'delta',
        'path': os.getenv('PS2_FAILURES_PATH', 's3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/tables/b1fe3620-8001-4c01-8ac9-17d7429973c2'),
        'logical_name': 'mars_dev.silver.device_failures',
    },
    'maintenance': {
        'format': 'delta',
        'path': os.getenv('PS2_MAINTENANCE_PATH', 's3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/tables/6acdc407-9e29-4cf7-a9dc-858c36e3799d'),
        'logical_name': 'mars_dev.silver.maintenance_ledger',
    },
    'dim_device': {
        'format': 'delta',
        'path': os.getenv('PS2_DIM_DEVICE_PATH', 's3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/tables/30038b0e-4d2b-42fd-a55d-fbc189884978'),
        'logical_name': 'mars_dev.silver.dim_device',
    },
    'hardware': {
        'format': 'delta',
        'path': os.getenv('PS2_HARDWARE_PATH', 's3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/tables/8ebee855-4b5f-47d3-8216-a00c80d18ffa'),
        'logical_name': 'mars_dev.silver.hw_config_current',
    },
    'chains': {
        'format': 'delta',
        'path': os.getenv('PS2_CHAINS_PATH', 's3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/__unitystorage/schemas/c28a7ed3-5f4d-408e-8557-da14c62143e5/tables/b333ea2a-0f1c-4bab-b0c1-96c9715c0aa6'),
        'logical_name': 'mars_dev.gold.device_ps2_chains',
    },
    'ps1': {
        'format': 'delta',
        'path': os.getenv('PS2_PS1_PATH', 's3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/__unitystorage/schemas/c28a7ed3-5f4d-408e-8557-da14c62143e5/tables/bbd02498-48e7-46f2-93c0-b8fcad810ff4'),
        'logical_name': 'mars_dev.gold.device_ps1_daily',
    },
    'ps3': {
        'format': 'delta',
        'path': os.getenv('PS2_PS3_PATH', 's3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/__unitystorage/schemas/c28a7ed3-5f4d-408e-8557-da14c62143e5/tables/757c240d-9294-4594-820d-449c7f2c9bf3'),
        'logical_name': 'mars_dev.gold.device_ps3_incident',
    },
    'ps4': {
        'format': 'delta',
        'path': os.getenv('PS2_PS4_PATH', 's3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/__unitystorage/schemas/c28a7ed3-5f4d-408e-8557-da14c62143e5/tables/2a3faa57-b342-4cbc-83c0-e819d21da48d'),
        'logical_name': 'mars_dev.gold.device_ps4_hourly',
    },
}

# Prediction fields emitted by PS1 Cell 24. They are evaluated against the direct
# Silver three-day label only when the optional cross-wired output is available.
PS1_PREDICTION_PROB_COL = os.getenv('PS2_PS1_PREDICTION_PROB_COL', 'ps1_fail_prob')
PS1_PREDICTION_FLAG_COL = os.getenv('PS2_PS1_PREDICTION_FLAG_COL', 'ps1_predicted')
PS1_PREDICTION_THRESHOLD_COL = os.getenv('PS2_PS1_PREDICTION_THRESHOLD_COL', 'threshold_used')
PS1_HORIZON_DAYS = LABEL_HORIZON_DAYS

LABEL_CONTRACT = {
    'target': PS1_LABEL_COL,
    'source': f'{PS1_SILVER_ROOT}/device_event_enriched/',
    'device_scope': 'is_current=true in direct S3 Silver dim_device',
    'positive_event': 'is_hardware_oos_event=true AND EVENT_STATE_TYPE_NAME=Set',
    'lookahead_calendar_days': list(range(1, LABEL_HORIZON_DAYS + 1)),
    'same_day_excluded': True,
    'right_censoring': f'exclude final {LABEL_HORIZON_DAYS + LABEL_DATA_LAG_DAYS} source days',
}

print(json.dumps({
    'run_id': RUN_ID, 'run_ts_utc': RUN_TS.isoformat(), 'notebook_version': NOTEBOOK_VERSION,
    'run_mode': RUN_MODE, 'production_mode': PRODUCTION_MODE, 'output_scope': OUTPUT_SCOPE,
    'fail_fast_on_critical_gate': FAIL_FAST_ON_CRITICAL_GATE,
    'lookback_days': ANALYSIS_LOOKBACK_DAYS, 'recent_days': RECENT_WINDOW_DAYS,
    'ps1_label_contract': LABEL_CONTRACT,
    'ps1_predictions_root': PS1_PREDICTIONS_ROOT,
    'spark_local_dir': str(SPARK_LOCAL_DIR),
    'spark_free_gib_at_start': round(SPARK_FREE_GIB_AT_START, 1),
    'spark_scratch_candidates': scratch_candidates,
    'export': f's3://{ARTIFACT_BUCKET}/{EXPORT_PREFIX}'
}, indent=2))



In [ ]:
# ── Cell 2: validated Spark + Delta + S3 runtime ──────────────────────────────
runtime_state_required = [
    'RUNTIME_PROFILE', 'JAVA_VERSION_LINE', 'RUN_ID', 'SPARK_DRIVER_MEMORY',
    'SPARK_DRIVER_MAX_RESULT_SIZE', 'SPARK_MASTER', 'SPARK_SHUFFLE_PARTITIONS',
    'SPARK_LOCAL_DIR',
]
missing_runtime_state = [name for name in runtime_state_required if name not in globals()]
if missing_runtime_state:
    raise RuntimeError(
        'Cell 2 requires the runtime bootstrap and configuration. Run Cell 0 and Cell 1 '
        f'first, or use Run All. Missing state: {", ".join(missing_runtime_state)}'
    )

try:
    import pyspark
    from pyspark.sql import SparkSession, Window, functions as F, types as T
    from pyspark import StorageLevel
    from delta import configure_spark_with_delta_pip
except ImportError as exc:
    raise RuntimeError(
        'PS2 runtime imports are unavailable after Cell 0. Restart the kernel and use Run All; '
        'for a scheduled job, bake the pinned packages into the SageMaker image.'
    ) from exc

# A pip PySpark distribution normally includes the Hadoop client but not hadoop-aws.
# Resolve the matching hadoop-aws artifact instead of hard-coding a version that can
# conflict with the active Spark distribution.
pyspark_jar_dir = Path(pyspark.__file__).resolve().parent / 'jars'
hadoop_client_jars = (
    sorted(pyspark_jar_dir.glob('hadoop-client-runtime-*.jar')) or
    sorted(pyspark_jar_dir.glob('hadoop-client-api-*.jar'))
)
hadoop_aws_jars = sorted(pyspark_jar_dir.glob('hadoop-aws-*.jar'))


def hadoop_version_from_jar(path):
    match = re.search(r'hadoop-client-(?:runtime|api)-(.+)\.jar$', path.name)
    return match.group(1) if match else None


HADOOP_VERSION = hadoop_version_from_jar(hadoop_client_jars[0]) if hadoop_client_jars else None
extra_maven_packages = []
if not hadoop_aws_jars:
    if not HADOOP_VERSION:
        raise RuntimeError(
            f'Cannot identify the Hadoop version in {pyspark_jar_dir}; '
            'PS2 cannot select a compatible S3 connector safely.'
        )
    extra_maven_packages.append(f'org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}')

active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

spark_builder = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName(f'ps2-failure-patterns-{RUN_ID[:8]}')
    .config('spark.driver.memory', SPARK_DRIVER_MEMORY)
    .config('spark.driver.maxResultSize', SPARK_DRIVER_MAX_RESULT_SIZE)
    .config('spark.local.dir', str(SPARK_LOCAL_DIR))
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.shuffle.partitions', str(SPARK_SHUFFLE_PARTITIONS))
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.adaptive.skewJoin.enabled', 'true')
    .config('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64m')
    .config('spark.sql.files.maxPartitionBytes', '64m')
    .config('spark.shuffle.compress', 'true')
    .config('spark.shuffle.spill.compress', 'true')
    .config('spark.io.compression.codec', 'zstd')
    .config('spark.memory.fraction', '0.55')
    .config('spark.memory.storageFraction', '0.20')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.selfDestruct.enabled', 'true')
)

try:
    spark = configure_spark_with_delta_pip(
        spark_builder, extra_packages=extra_maven_packages
    ).getOrCreate()
except Exception as exc:
    raise RuntimeError(
        'Spark/Delta startup failed. Check the Java runtime and outbound access to Maven '
        'Central for the first JAR resolution. Scheduled production runs should use an image '
        'with the Delta and hadoop-aws JARs already cached.'
    ) from exc

try:
    class_loader = spark._jvm.java.lang.Thread.currentThread().getContextClassLoader()
    class_loader.loadClass('io.delta.sql.DeltaSparkSessionExtension')
    class_loader.loadClass('org.apache.hadoop.fs.s3a.S3AFileSystem')
    spark.sql('SELECT 1 AS runtime_probe').collect()
except Exception as exc:
    spark.stop()
    raise RuntimeError(
        'Spark started but the Delta Lake or S3A connector was not usable. Review the Ivy '
        'resolution immediately above this error; do not continue to the source or publish cells.'
    ) from exc

try:
    import boto3
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError as exc:
    raise RuntimeError('Missing a lightweight reporting dependency: boto3, pandas, matplotlib, seaborn.') from exc

s3 = boto3.client('s3')
print(
    'Spark:', spark.version,
    '| Delta:', RUNTIME_PROFILE['delta_spark'],
    '| Hadoop:', HADOOP_VERSION,
    '| Master:', SPARK_MASTER,
    '| Driver heap:', SPARK_DRIVER_MEMORY,
    '| Shuffle partitions:', SPARK_SHUFFLE_PARTITIONS,
    '| Scratch:', str(SPARK_LOCAL_DIR),
    '| Scratch free GiB:', f'{disk_free_gib(SPARK_LOCAL_DIR):.1f}',
    '| applicationId:', spark.sparkContext.applicationId,
)



In [ ]:
# ── Cell 3: source contracts and stable, case-insensitive readers ────────────
def _casefold_column_map(columns, table_name):
    resolved = {}
    ambiguous = {}
    for actual in columns:
        folded = actual.casefold()
        if folded in resolved and resolved[folded] != actual:
            ambiguous.setdefault(folded, [resolved[folded]]).append(actual)
        else:
            resolved[folded] = actual
    if ambiguous:
        raise ValueError(f'{table_name} contains columns differing only by case: {ambiguous}')
    return resolved


def assert_columns(df, table_name, required):
    mapping = _casefold_column_map(df.columns, table_name)
    missing = sorted(c for c in required if c.casefold() not in mapping)
    if missing:
        raise ValueError(
            f'{table_name} violates the PS2 v2.5 input contract; missing columns: {missing}. '
            f'Observed columns: {df.columns}'
        )


def spark_storage_path(path):
    # Spark's supported S3 connector is S3A. Boto3 output locations remain s3:// URIs.
    return f's3a://{path[5:]}' if path.startswith('s3://') else path


def read_source(name, required, optional=()):
    """Read a Delta or Parquet source and project canonical names case-insensitively."""
    spec = SOURCE[name]
    load_path = spark_storage_path(spec['path'])
    try:
        df = spark.read.format(spec['format']).load(load_path)
    except Exception as exc:
        spark_message = str(exc)
        override_name = f'PS2_{name.upper()}_PATH'
        if '[PATH_NOT_FOUND]' in spark_message or 'path does not exist' in spark_message.lower():
            if spec['format'] == 'delta':
                diagnosis = (
                    'The configured Delta prefix does not exist. Unity Storage table UUIDs can '
                    f'change after a rebuild; obtain the current location and set {override_name}.'
                )
            else:
                diagnosis = (
                    'The stable direct-S3 Silver export is missing at this prefix. Confirm the '
                    f'PS1 export root or set {override_name}.'
                )
        elif any(marker in spark_message.lower() for marker in ('accessdenied', 'access denied', 'forbidden', 'status code: 403')):
            diagnosis = 'The path may exist, but the SageMaker execution role cannot read it.'
        else:
            diagnosis = 'Validate the location, IAM permissions, S3 endpoint, format, and Spark S3A runtime.'
        raise RuntimeError(
            f'Cannot read {spec["logical_name"]} from its configured path. {diagnosis} '
            f'Configured path: {spec["path"]}; Spark path: {load_path}; '
            f'environment override: {override_name}'
        ) from exc

    assert_columns(df, spec['logical_name'], required)
    mapping = _casefold_column_map(df.columns, spec['logical_name'])
    requested = list(dict.fromkeys(list(required) + list(optional)))
    selected = [
        F.col(mapping[c.casefold()]).alias(c)
        for c in requested
        if c.casefold() in mapping
    ]
    return df.select(*selected)


def truthy(col):
    return F.coalesce(col.cast('boolean'), F.lit(False))


def canonical_string(col, fallback='UNKNOWN'):
    return F.coalesce(F.trim(col.cast('string')), F.lit(fallback))


def canonical_device_category(col):
    normalized = F.upper(F.regexp_replace(F.trim(col.cast('string')), r'\s+', ' '))
    # Non-empty, non-target categories are valid OTHER scope, not mapping failures.
    result = F.lit('OTHER')
    for raw_name, canonical_name in reversed(list(DEVICE_CATEGORY_ALIASES.items())):
        raw_normalized = raw_name.replace('_', ' ').upper()
        result = F.when(
            F.regexp_replace(normalized, '_', ' ') == F.lit(raw_normalized),
            F.lit(canonical_name),
        ).otherwise(result)
    return F.when(col.isNull() | (F.trim(col.cast('string')) == ''), F.lit('UNKNOWN')).otherwise(result)


def compact_count(df, label):
    n = df.count()
    print(f'{label}: {n:,} rows')
    return n


def single_row_dict(df):
    return df.first().asDict(recursive=True)


EVENT_REQUIRED = [
    'DW_DEVICE_EVENT_ID', 'EVENT_DTM', 'DEVICE_ID', 'DEVICE_KEY', 'EVENT_STATE_TYPE_NAME',
    'CLEAR_DTM', 'transit_day', 'COMPONENT_TYPE_NAME', 'COMPONENT_SERIAL_NBR',
    'COMPONENT_POSITION', 'mars_device_category', 'FACILITY_ID', 'FACILITY_NAME', 'BUS_ID',
    'EVENT_TYPE_ID', 'EVENT_TYPE_NAME', 'component_subsystem', 'severity', 'event_priority',
    'is_oos_event', 'is_hardware_oos_event', 'is_commanded_oos_event', 'is_set_clear',
    'requires_service_call'
]
LABEL_EVENT_REQUIRED = [
    'DEVICE_KEY', 'DEVICE_ID', 'EVENT_DTM', 'EVENT_STATE_TYPE_NAME',
    'is_hardware_oos_event', 'mars_device_category',
]
LABEL_EVENT_OPTIONAL = [
    'DW_DEVICE_EVENT_ID', 'is_commanded_oos_event', 'is_oos_event', 'CLEAR_DTM',
]
LABEL_DIM_REQUIRED = ['DEVICE_KEY', 'DEVICE_ID', 'is_current', 'mars_device_category']
LABEL_DIM_OPTIONAL = ['is_active']
OUTAGE_REQUIRED = [
    'source_event_id', 'DEVICE_ID', 'outage_start', 'outage_end', 'duration_min',
    'is_chargeable', 'failure_level', 'component_subsystem', 'COMPONENT_SERIAL_NBR',
    'mars_device_category', 'FACILITY_ID', 'BUS_ID'
]
FAILURE_REQUIRED = ['DEVICE_ID', 'failure_date', 'failure_event_count', 'first_failure_dtm', 'failure_source']
MAINT_REQUIRED = [
    'DEVICE_ID', 'ledger_date', 'event_dtm', 'event_end_dtm', 'ledger_type',
    'component_subsystem', 'is_commanded_oos', 'is_maintenance_oos', 'source_event_id'
]
DIM_DEVICE_REQUIRED = [
    'DEVICE_ID', 'TRANSIT_ARRAY_ID', 'ARRAY_POSITION', 'FACILITY_ID',
    'mars_device_category', 'is_current', 'is_active'
]
HARDWARE_REQUIRED = [
    'DEVICE_ID', 'COMPONENT_SERIAL_NBR', 'COMPONENT_DESCRIPTION',
    'component_age_days', 'hw_source'
]
CHAINS_REQUIRED = [
    'DEVICE_ID', 'transit_day', 'is_coordinated_station_failure',
    'is_major_station_event'
]
PS1_REQUIRED = ['DEVICE_ID', 'transit_day']
PS1_OPTIONAL = [
    'DEVICE_KEY', 'mars_device_category', 'tap_count', 'daily_revenue_cents',
    PS1_LABEL_COL, PS1_LEGACY_SLA_LABEL_COL,
]
PS3_REQUIRED = ['device_id', 'transit_day', 'is_device_fault', 'is_chargeable', 'failure_level']
PS4_REQUIRED = ['DEVICE_ID', 'transit_day', 'ensemble_anomaly_flag']

SOURCE_PREFLIGHT_CONTRACTS = {
    'label_events': (LABEL_EVENT_REQUIRED, LABEL_EVENT_OPTIONAL),
    'label_dim_device': (LABEL_DIM_REQUIRED, LABEL_DIM_OPTIONAL),
    'events': (EVENT_REQUIRED, ()),
    'outage': (OUTAGE_REQUIRED, ()),
    'failures': (FAILURE_REQUIRED, ()),
    'maintenance': (MAINT_REQUIRED, ()),
    'dim_device': (DIM_DEVICE_REQUIRED, ()),
    'hardware': (HARDWARE_REQUIRED, ()),
    'chains': (CHAINS_REQUIRED, ()),
    'ps1': (PS1_REQUIRED, PS1_OPTIONAL),
    'ps3': (PS3_REQUIRED, ()),
    'ps4': (PS4_REQUIRED, ()),
}


def preflight_source_contracts():
    """Resolve every location and required schema before any large Spark action."""
    frames = {}
    summary = {}
    for name, (required, optional) in SOURCE_PREFLIGHT_CONTRACTS.items():
        frames[name] = read_source(name, required, optional)
        summary[name] = {
            'logical_name': SOURCE[name]['logical_name'],
            'format': SOURCE[name]['format'],
            'required_columns': len(required),
            'selected_columns': len(frames[name].columns),
            'status': 'PASS',
        }
    print(json.dumps({'source_preflight': 'PASS', 'sources': summary}, indent=2))
    return frames


In [ ]:
# ── Cell 4: load narrow inputs and establish the real as-of boundary ──────────
SOURCE_FRAMES = preflight_source_contracts()
events_all = SOURCE_FRAMES['events']
events_target_all = (
    events_all
    .withColumn('_target_device_category', canonical_device_category(F.col('mars_device_category')))
    .where(F.col('_target_device_category').isin(*TARGET_DEVICE_CATEGORIES))
)
source_max_ts = events_target_all.agg(F.max('EVENT_DTM').alias('max_event_ts')).first()['max_event_ts']
if source_max_ts is None:
    raise RuntimeError('device_event_enriched has no EVENT_DTM values; PS2 cannot establish an as-of boundary.')

if AS_OF_DATE_OVERRIDE:
    requested_as_of = dt.datetime.fromisoformat(AS_OF_DATE_OVERRIDE).replace(tzinfo=None)
    as_of_ts = min(requested_as_of, source_max_ts)
else:
    as_of_ts = source_max_ts

analysis_start_ts = as_of_ts - dt.timedelta(days=ANALYSIS_LOOKBACK_DAYS)
recent_start_ts = as_of_ts - dt.timedelta(days=RECENT_WINDOW_DAYS)
source_lag_days = (RUN_TS.replace(tzinfo=None) - source_max_ts).total_seconds() / 86400.0
print(f'Analysis window: {analysis_start_ts} -> {as_of_ts}; source event freshness lag={source_lag_days:.1f} days')
if source_lag_days > MAX_SOURCE_LAG_DAYS:
    disposition = (
        'BLOCKED at the production quality gate' if PRODUCTION_MODE
        else f'recorded as a {RUN_MODE.lower()} warning'
    )
    print(
        f'WARNING: source data is {source_lag_days:.1f} days behind run time '
        f'(threshold={MAX_SOURCE_LAG_DAYS}); this run will be {disposition}.'
    )

events_raw = (
    events_all
    .where((F.col('EVENT_DTM') >= F.lit(analysis_start_ts)) & (F.col('EVENT_DTM') <= F.lit(as_of_ts)))
    .withColumnRenamed('DW_DEVICE_EVENT_ID', 'event_id')
    .withColumnRenamed('EVENT_DTM', 'event_ts')
    .withColumnRenamed('DEVICE_ID', 'device_id')
    .withColumnRenamed('DEVICE_KEY', 'device_key')
    .withColumnRenamed('mars_device_category', 'device_category')
    .withColumnRenamed('COMPONENT_TYPE_NAME', 'component_type_name')
    .withColumnRenamed('COMPONENT_SERIAL_NBR', 'component_serial_nbr')
    .withColumnRenamed('COMPONENT_POSITION', 'component_position')
    .withColumnRenamed('EVENT_TYPE_ID', 'event_type_id')
    .withColumnRenamed('EVENT_TYPE_NAME', 'event_type_name')
    .withColumn('device_id', F.col('device_id').cast('string'))
    .withColumn('device_key', F.col('device_key').cast('string'))
    .withColumn('event_date', F.to_date('event_ts'))
    .withColumn('component_subsystem', canonical_string(F.col('component_subsystem')))
    .withColumn('device_category_raw', canonical_string(F.col('device_category')))
    .withColumn('device_category', canonical_device_category(F.col('device_category')))
    .filter(F.col('device_id').isNotNull() & F.col('event_ts').isNotNull())
    .withColumn(
        'event_id',
        F.coalesce(
            F.col('event_id').cast('string'),
            F.concat(
                F.lit('SYNTHETIC|'),
                F.sha2(F.concat_ws(
                    '||', F.col('device_id').cast('string'), F.col('event_ts').cast('string'),
                    F.coalesce(F.col('event_type_id').cast('string'), F.lit('')),
                    F.coalesce(F.col('component_type_name').cast('string'), F.lit('')),
                    F.coalesce(F.col('component_position').cast('string'), F.lit('')),
                ), 256),
            ),
        ),
    )
)

pre_dedupe_n = events_raw.count()
# `event_id` is the governed uniqueness key. Profile category mapping before applying the
# TVM/GATE/VALIDATOR analytical scope so legitimate out-of-scope devices remain visible.
events_dedup_all = events_raw.dropDuplicates(['event_id']).persist(StorageLevel.DISK_ONLY)
post_dedupe_all_n = events_dedup_all.count()

category_profile = (
    events_dedup_all
    .groupBy('device_category_raw', 'device_category')
    .agg(
        F.count('*').alias('event_count'),
        F.countDistinct('device_id').alias('device_count'),
        F.min('event_ts').alias('first_event_ts'),
        F.max('event_ts').alias('last_event_ts'),
    )
    .withColumn('is_mapped', F.col('device_category') != 'UNKNOWN')
    .withColumn('is_target_scope', F.col('device_category').isin(*TARGET_DEVICE_CATEGORIES))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
category_profile_row_count = category_profile.count()

events_raw = (
    events_dedup_all
    .where(F.col('device_category').isin(*TARGET_DEVICE_CATEGORIES))
    .persist(StorageLevel.DISK_ONLY)
)
post_dedupe_n = events_raw.count()
print(
    f'Event rows: raw={pre_dedupe_n:,}, deduplicated_all={post_dedupe_all_n:,}, '
    f'target_scope={post_dedupe_n:,}, removed_duplicates={pre_dedupe_n - post_dedupe_all_n:,}, '
    f'excluded_out_of_scope={post_dedupe_all_n - post_dedupe_n:,}; '
    f'category_profile_rows={category_profile_row_count:,}'
)
# Both compact consumers are now materialized; release the all-family dedupe cache.
events_dedup_all.unpersist(blocking=True)

outage = (
    SOURCE_FRAMES['outage']
    .where((F.col('outage_start') >= F.lit(analysis_start_ts - dt.timedelta(days=1))) & (F.col('outage_start') <= F.lit(as_of_ts)))
    .select(
        F.col('source_event_id').cast('string').alias('event_id'), F.col('DEVICE_ID').cast('string').alias('outage_device_id'),
        'outage_start', 'outage_end', F.col('duration_min').cast('double').alias('outage_duration_min'),
        truthy(F.col('is_chargeable')).alias('outage_is_chargeable'), F.col('failure_level').cast('double').alias('outage_failure_level'),
        canonical_string(F.col('component_subsystem')).alias('outage_component_subsystem'),
        F.col('COMPONENT_SERIAL_NBR').cast('string').alias('outage_component_serial_nbr')
    )
)
outage_value_columns = [column for column in outage.columns if column != 'event_id']
outage = (
    outage.groupBy('event_id')
    .agg(F.max_by(
        F.struct(*[F.col(column) for column in outage_value_columns]),
        F.col('outage_start'),
    ).alias('_latest'))
    .select('event_id', *[
        F.col(f'_latest.{column}').alias(column) for column in outage_value_columns
    ])
)

failure_daily = (
    SOURCE_FRAMES['failures']
    .where((F.col('failure_date') >= F.to_date(F.lit(analysis_start_ts))) & (F.col('failure_date') <= F.to_date(F.lit(as_of_ts))))
    .select(
        F.col('DEVICE_ID').cast('string').alias('failure_device_id'), F.col('failure_date').alias('failure_date'),
        F.col('failure_event_count').cast('long').alias('failure_event_count'),
        F.col('first_failure_dtm').alias('first_failure_dtm'), F.col('failure_source').cast('string').alias('failure_source')
    )
    .groupBy('failure_device_id', 'failure_date')
    .agg(
        F.sum('failure_event_count').alias('validated_failure_events_same_day'),
        F.min('first_failure_dtm').alias('first_failure_dtm'),
        F.first('failure_source', ignorenulls=True).alias('failure_source')
    )
)

maintenance = (
    SOURCE_FRAMES['maintenance']
    .where((F.coalesce(F.col('event_dtm'), F.to_timestamp('ledger_date')) >= F.lit(analysis_start_ts - dt.timedelta(days=2))) &
           (F.coalesce(F.col('event_dtm'), F.to_timestamp('ledger_date')) <= F.lit(as_of_ts)))
    .select(
        F.col('DEVICE_ID').cast('string').alias('maintenance_device_id'), F.col('ledger_date').alias('maintenance_date'),
        F.coalesce(F.col('event_dtm'), F.to_timestamp('ledger_date')).alias('maintenance_start_ts'),
        F.coalesce(F.col('event_end_dtm'), F.expr("event_dtm + INTERVAL 1 DAYS"), F.to_timestamp('ledger_date') + F.expr('INTERVAL 1 DAYS')).alias('maintenance_end_ts'),
        canonical_string(F.col('ledger_type')).alias('ledger_type'), canonical_string(F.col('component_subsystem')).alias('maintenance_component_subsystem'),
        truthy(F.col('is_commanded_oos')).alias('maintenance_commanded_oos'),
        truthy(F.col('is_maintenance_oos')).alias('maintenance_oos'), F.col('source_event_id').cast('string').alias('maintenance_source_event_id')
    )
)

# `dim_device` is used only for the stable, current asset topology (gate array / position)
# and active-device exposure. Event-side facility/category values remain authoritative for
# historical events and are never overwritten by a current SCD row.
dim_current = (
    SOURCE_FRAMES['dim_device']
    .where(truthy(F.col('is_current')))
    .select(
        F.col('DEVICE_ID').cast('string').alias('dim_device_id'), F.col('TRANSIT_ARRAY_ID').cast('string').alias('transit_array_id'),
        F.col('ARRAY_POSITION').cast('string').alias('array_position'), F.col('FACILITY_ID').cast('string').alias('dim_facility_id'),
        canonical_device_category(F.col('mars_device_category')).alias('dim_device_category'), truthy(F.col('is_active')).alias('is_active_current')
    )
    .dropDuplicates(['dim_device_id'])
)

# This is a current-configuration enrichment only. It supplies age/context for a component
# serial but never rewrites historical event locations or claims that age was known at an
# earlier event timestamp.
hardware_current = (
    SOURCE_FRAMES['hardware']
    .select(
        F.col('DEVICE_ID').cast('string').alias('hardware_device_id'),
        F.when(F.trim(F.col('COMPONENT_SERIAL_NBR').cast('string')) != '', F.trim(F.col('COMPONENT_SERIAL_NBR').cast('string'))).alias('hardware_component_serial_id'),
        canonical_string(F.col('COMPONENT_DESCRIPTION')).alias('hardware_component_description'),
        F.col('component_age_days').cast('double').alias('current_component_age_days'),
        canonical_string(F.col('hw_source')).alias('hardware_source')
    )
    .where(F.col('hardware_component_serial_id').isNotNull())
    .groupBy('hardware_component_serial_id')
    .agg(
        F.max('current_component_age_days').alias('current_component_age_days'),
        F.first('hardware_component_description', ignorenulls=True).alias('hardware_component_description'),
        F.first('hardware_source', ignorenulls=True).alias('hardware_source'),
        F.countDistinct('hardware_device_id').alias('current_config_device_count')
    )
)

# Existing PS2 chains are used only as a station/common-cause enrichment.  Their raw
# `oos_in_chain` values never create an OOS label in this notebook.
chains = (
    SOURCE_FRAMES['chains']
    .where((F.col('transit_day') >= F.to_date(F.lit(analysis_start_ts))) & (F.col('transit_day') <= F.to_date(F.lit(as_of_ts))))
    .select(F.col('DEVICE_ID').cast('string').alias('device_id'), F.col('transit_day').alias('event_date'),
            truthy(F.col('is_coordinated_station_failure')).alias('coordinated_station_flag'),
            truthy(F.col('is_major_station_event')).alias('major_station_flag'))
    .groupBy('device_id', 'event_date').agg(F.max('coordinated_station_flag').alias('coordinated_station_flag'), F.max('major_station_flag').alias('major_station_flag'))
)



In [ ]:
# ── Cell 5: direct-S3 Silver PS1-compatible three-day failure label ─────────
# This deliberately reproduces PS1's label, including the current-device restriction and
# exclusion of the failure day itself. It is a prediction target, not the governed PS2
# operational onset label used by the causal pattern tests.

label_events_source = SOURCE_FRAMES['label_events']
label_source_max_ts = (
    label_events_source
    .where(canonical_device_category(F.col('mars_device_category')).isin(*TARGET_DEVICE_CATEGORIES))
    .agg(F.max('EVENT_DTM').alias('label_source_max_ts'))
    .first()['label_source_max_ts']
)
if label_source_max_ts is None:
    raise RuntimeError('Direct S3 Silver label source has no target-family EVENT_DTM values.')
label_as_of_ts = min(as_of_ts, label_source_max_ts)
direct_label_source_lag_days = max(
    0.0, (as_of_ts - label_source_max_ts).total_seconds() / 86400.0
)
label_cutoff_date = (
    label_as_of_ts.date() - dt.timedelta(days=LABEL_HORIZON_DAYS + LABEL_DATA_LAG_DAYS)
)
label_start_date = analysis_start_ts.date()
if label_cutoff_date < label_start_date:
    raise RuntimeError(
        'The source window is shorter than the configured PS1 label horizon/data lag.'
    )

label_dim = (
    SOURCE_FRAMES['label_dim_device']
    .where(truthy(F.col('is_current')))
    .select(
        F.col('DEVICE_KEY').cast('string').alias('label_device_key'),
        F.col('DEVICE_ID').cast('string').alias('label_device_id'),
        canonical_device_category(F.col('mars_device_category')).alias('label_device_category'),
        (truthy(F.col('is_active')) if 'is_active' in SOURCE_FRAMES['label_dim_device'].columns else F.lit(True)).alias('label_is_active'),
    )
    .where(F.col('label_device_key').isNotNull() & F.col('label_device_id').isNotNull())
    .where(F.col('label_device_category').isin(*TARGET_DEVICE_CATEGORIES))
    .dropDuplicates(['label_device_key', 'label_device_id'])
    .persist(StorageLevel.DISK_ONLY)
)

label_device_map = (
    label_dim.groupBy('label_device_id')
    .agg(
        F.first('label_device_key', ignorenulls=True).alias('label_device_key'),
        F.first('label_device_category', ignorenulls=True).alias('label_device_category'),
        F.max(F.col('label_is_active').cast('int')).cast('boolean').alias('label_is_active'),
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
label_device_count = label_device_map.count()

ps1_source = SOURCE_FRAMES['ps1']
ps1_category_expr = (
    canonical_device_category(F.col('mars_device_category'))
    if 'mars_device_category' in ps1_source.columns
    else F.lit(None).cast('string')
)
ps1_label_expr = (
    F.col(PS1_LABEL_COL).cast('byte')
    if PS1_LABEL_COL in ps1_source.columns
    else F.lit(None).cast('byte')
)
legacy_label_expr = (
    F.col(PS1_LEGACY_SLA_LABEL_COL).cast('byte')
    if PS1_LEGACY_SLA_LABEL_COL in ps1_source.columns
    else F.lit(None).cast('byte')
)

label_base_daily = (
    ps1_source
    .select(
        F.col('DEVICE_ID').cast('string').alias('device_id'),
        F.to_date('transit_day').alias('label_date'),
        ps1_category_expr.alias('_ps1_device_category'),
        ps1_label_expr.alias('source_ps1_hardware_oos_3d'),
        legacy_label_expr.alias('source_legacy_will_fail_3d'),
    )
    .join(label_device_map, F.col('device_id') == F.col('label_device_id'), 'inner')
    .withColumn(
        'device_category',
        F.when(
            F.col('_ps1_device_category').isin(*TARGET_DEVICE_CATEGORIES),
            F.col('_ps1_device_category'),
        ).otherwise(F.col('label_device_category')),
    )
    .where(
        (F.col('label_date') >= F.lit(label_start_date))
        & (F.col('label_date') <= F.lit(label_cutoff_date))
        & F.col('device_category').isin(*TARGET_DEVICE_CATEGORIES)
    )
    .groupBy('device_id', 'label_date', 'device_category')
    .agg(
        F.max('source_ps1_hardware_oos_3d').cast('byte').alias('source_ps1_hardware_oos_3d'),
        F.max('source_legacy_will_fail_3d').cast('byte').alias('source_legacy_will_fail_3d'),
    )
    .persist(StorageLevel.DISK_ONLY)
)

silver_event_id_expr = (
    F.coalesce(
        F.col('DW_DEVICE_EVENT_ID').cast('string'),
        F.sha2(
            F.concat_ws(
                '||', F.col('DEVICE_ID').cast('string'), F.col('EVENT_DTM').cast('string'),
                F.col('EVENT_STATE_TYPE_NAME').cast('string'),
            ),
            256,
        ),
    )
    if 'DW_DEVICE_EVENT_ID' in label_events_source.columns
    else F.sha2(
        F.concat_ws(
            '||', F.col('DEVICE_ID').cast('string'), F.col('EVENT_DTM').cast('string'),
            F.col('EVENT_STATE_TYPE_NAME').cast('string'),
        ),
        256,
    )
)
silver_commanded_expr = (
    truthy(F.col('is_commanded_oos_event'))
    if 'is_commanded_oos_event' in label_events_source.columns
    else F.lit(False)
)

raw_silver_hardware_oos_set = (
    label_events_source
    .where(truthy(F.col('is_hardware_oos_event')))
    .where(F.upper(F.trim(F.col('EVENT_STATE_TYPE_NAME').cast('string'))) == 'SET')
    .where(canonical_device_category(F.col('mars_device_category')).isin(*TARGET_DEVICE_CATEGORIES))
    .where(
        (F.to_date('EVENT_DTM') >= F.date_add(F.lit(label_start_date), 1))
        & (F.to_date('EVENT_DTM') <= F.date_add(F.lit(label_cutoff_date), LABEL_HORIZON_DAYS))
    )
    .select(
        F.col('DEVICE_KEY').cast('string').alias('silver_device_key'),
        F.col('DEVICE_ID').cast('string').alias('silver_device_id'),
        F.col('EVENT_DTM').cast('timestamp').alias('silver_failure_ts'),
        canonical_device_category(F.col('mars_device_category')).alias('silver_device_category'),
        silver_event_id_expr.alias('silver_failure_event_id'),
        silver_commanded_expr.alias('silver_commanded_oos_flag'),
    )
    .dropDuplicates(['silver_failure_event_id'])
    .persist(StorageLevel.DISK_ONLY)
)

raw_silver_label_event_count = raw_silver_hardware_oos_set.count()
silver_hardware_oos_set_events = (
    raw_silver_hardware_oos_set.alias('e')
    .join(
        F.broadcast(label_dim.select('label_device_key').distinct()).alias('d'),
        F.col('e.silver_device_key') == F.col('d.label_device_key'),
        'inner',
    )
    .select('e.*')
    .withColumnRenamed('silver_device_id', 'device_id')
    .withColumnRenamed('silver_device_category', 'device_category')
    .withColumn('failure_date', F.to_date('silver_failure_ts'))
    .persist(StorageLevel.DISK_ONLY)
)
linked_silver_label_event_count = silver_hardware_oos_set_events.count()
# The linked event set and compact device map are materialized; the two source-side
# dedupe caches are no longer needed for any later action.
raw_silver_hardware_oos_set.unpersist(blocking=True)
label_dim.unpersist(blocking=True)
silver_label_current_device_linkage = (
    linked_silver_label_event_count / max(raw_silver_label_event_count, 1)
)

label_horizon_sequence = spark.range(1, LABEL_HORIZON_DAYS + 1).select(
    F.col('id').cast('int').alias('lead_day')
)
expanded_label_events = (
    silver_hardware_oos_set_events.crossJoin(F.broadcast(label_horizon_sequence))
    .withColumn('label_date', F.date_sub('failure_date', F.col('lead_day')))
    .select(
        'device_id', 'device_category', 'label_date', 'lead_day',
        'silver_failure_ts', 'silver_failure_event_id', 'silver_commanded_oos_flag',
    )
)

positive_label_days = (
    expanded_label_events
    .groupBy('device_id', 'device_category', 'label_date')
    .agg(
        F.min('silver_failure_ts').alias('next_hardware_oos_set_ts'),
        F.countDistinct('silver_failure_event_id').alias('future_hardware_oos_set_events_3d'),
        F.min('lead_day').alias('days_to_next_hardware_oos_set'),
        F.max(F.col('silver_commanded_oos_flag').cast('int')).cast('boolean').alias('future_window_contains_commanded_oos'),
    )
)

failure_label_panel = (
    label_base_daily.alias('b')
    .join(
        positive_label_days.alias('p'),
        ['device_id', 'device_category', 'label_date'],
        'left',
    )
    .withColumn(
        PS1_LABEL_COL,
        F.when(F.col('next_hardware_oos_set_ts').isNotNull(), F.lit(1)).otherwise(F.lit(0)).cast('byte'),
    )
    .withColumn(
        'hours_to_next_hardware_oos_set',
        F.when(
            F.col('next_hardware_oos_set_ts').isNotNull(),
            (F.unix_timestamp('next_hardware_oos_set_ts') - F.unix_timestamp(F.to_timestamp('label_date'))) / 3600.0,
        ),
    )
    .fillna(0, subset=['future_hardware_oos_set_events_3d'])
    .withColumn('label_is_right_censored', F.lit(False))
    .persist(StorageLevel.DISK_ONLY)
)

failure_label_daily = (
    failure_label_panel
    .groupBy('label_date', 'device_category')
    .agg(
        F.count('*').alias('eligible_device_days'),
        F.sum(F.col(PS1_LABEL_COL).cast('long')).alias('positive_device_days'),
        F.countDistinct('device_id').alias('eligible_devices'),
        F.countDistinct(F.when(F.col(PS1_LABEL_COL) == 1, F.col('device_id'))).alias('positive_devices'),
        F.sum('future_hardware_oos_set_events_3d').alias('future_hardware_oos_set_events'),
        F.expr('percentile_approx(hours_to_next_hardware_oos_set, 0.50)').alias('median_hours_to_next_oos'),
        F.expr('percentile_approx(hours_to_next_hardware_oos_set, 0.90)').alias('p90_hours_to_next_oos'),
    )
    .withColumn('negative_device_days', F.col('eligible_device_days') - F.col('positive_device_days'))
    .withColumn('label_positive_rate', F.col('positive_device_days') / F.greatest(F.col('eligible_device_days'), F.lit(1)))
)

def _label_summary(frame, category_value=None):
    scoped = frame if category_value is None else frame.where(F.col('device_category') == category_value)
    output_category = 'ALL' if category_value is None else category_value
    return scoped.groupBy().agg(
        F.count('*').alias('eligible_device_days'),
        F.sum(F.col(PS1_LABEL_COL).cast('long')).alias('positive_device_days'),
        F.countDistinct('device_id').alias('eligible_devices'),
        F.countDistinct(F.when(F.col(PS1_LABEL_COL) == 1, F.col('device_id'))).alias('positive_devices'),
        F.sum('future_hardware_oos_set_events_3d').alias('future_hardware_oos_set_events'),
        F.avg('hours_to_next_hardware_oos_set').alias('mean_hours_to_next_oos'),
        F.expr('percentile_approx(hours_to_next_hardware_oos_set, 0.50)').alias('median_hours_to_next_oos'),
        F.expr('percentile_approx(hours_to_next_hardware_oos_set, 0.90)').alias('p90_hours_to_next_oos'),
        F.sum(F.when(F.col('future_window_contains_commanded_oos'), 1).otherwise(0)).alias('positive_days_with_commanded_oos'),
    ).withColumn('device_category', F.lit(output_category))


failure_label_summary = _label_summary(failure_label_panel).unionByName(
    _label_summary(failure_label_panel, TARGET_DEVICE_CATEGORIES[0])
).unionByName(
    _label_summary(failure_label_panel, TARGET_DEVICE_CATEGORIES[1])
).unionByName(
    _label_summary(failure_label_panel, TARGET_DEVICE_CATEGORIES[2])
).withColumn(
    'negative_device_days', F.col('eligible_device_days') - F.col('positive_device_days')
).withColumn(
    'label_positive_rate', F.col('positive_device_days') / F.greatest(F.col('eligible_device_days'), F.lit(1))
).withColumn(
    'positive_device_share', F.col('positive_devices') / F.greatest(F.col('eligible_devices'), F.lit(1))
).withColumn(
    'label_horizon_days', F.lit(LABEL_HORIZON_DAYS)
).withColumn(
    'label_cutoff_date', F.lit(label_cutoff_date).cast('date')
).withColumn(
    'label_definition', F.lit('DIRECT_S3_SILVER_HW_OOS_SET_FUTURE_CALENDAR_DAYS_1_TO_3_CURRENT_DEVICE')
)

eligible_by_category = failure_label_panel.groupBy('device_category').agg(
    F.count('*').alias('eligible_device_days')
)
failure_horizon_profile = (
    expanded_label_events
    .join(
        label_base_daily.select('device_id', 'device_category', 'label_date'),
        ['device_id', 'device_category', 'label_date'],
        'inner',
    )
    .select('device_id', 'device_category', 'label_date', 'lead_day')
    .dropDuplicates()
    .groupBy('device_category', 'lead_day')
    .agg(F.count('*').alias('positive_device_days_at_lead'))
    .join(eligible_by_category, 'device_category', 'left')
    .withColumn(
        'positive_rate_at_lead',
        F.col('positive_device_days_at_lead') / F.greatest(F.col('eligible_device_days'), F.lit(1)),
    )
    .withColumn('label_definition', F.lit('FAILURE_ON_EXACT_FUTURE_CALENDAR_DAY'))
)

parity_by_category = (
    failure_label_panel.groupBy('device_category')
    .agg(
        F.count('*').alias('eligible_device_days'),
        F.sum(F.when(F.col('source_ps1_hardware_oos_3d').isNotNull(), 1).otherwise(0)).alias('comparable_device_days'),
        F.sum(F.when(
            F.col('source_ps1_hardware_oos_3d').isNotNull()
            & (F.col('source_ps1_hardware_oos_3d') == F.col(PS1_LABEL_COL)), 1,
        ).otherwise(0)).alias('matching_device_days'),
        F.sum(F.when(
            F.col('source_ps1_hardware_oos_3d').isNotNull()
            & (F.col('source_ps1_hardware_oos_3d') != F.col(PS1_LABEL_COL)), 1,
        ).otherwise(0)).alias('mismatching_device_days'),
        F.avg(F.col('source_ps1_hardware_oos_3d').cast('double')).alias('source_label_positive_rate'),
        F.avg(F.col(PS1_LABEL_COL).cast('double')).alias('rebuilt_label_positive_rate'),
        F.avg(F.col('source_legacy_will_fail_3d').cast('double')).alias('legacy_sla_positive_rate'),
    )
    .withColumn(
        'parity_rate',
        F.when(F.col('comparable_device_days') > 0, F.col('matching_device_days') / F.col('comparable_device_days')),
    )
    .withColumn(
        'parity_status',
        F.when(F.col('comparable_device_days') == 0, 'SOURCE_LABEL_NOT_PERSISTED')
        .when(F.col('parity_rate') >= 0.999, 'PARITY_PASS')
        .otherwise('PARITY_REVIEW'),
    )
    .withColumn('rebuilt_target', F.lit(PS1_LABEL_COL))
)
ps1_label_parity = parity_by_category

label_counts_direct = single_row_dict(failure_label_panel.agg(
    F.count('*').alias('eligible_device_days'),
    F.sum(F.col(PS1_LABEL_COL).cast('long')).alias('positive_device_days'),
    F.countDistinct('device_id').alias('eligible_devices'),
    F.countDistinct('device_category').alias('label_device_categories'),
))
print('Direct-Silver PS1 label counts:', label_counts_direct)
print(
    f'Direct Silver HW-OOS Set events: raw={raw_silver_label_event_count:,}, '
    f'current-device-linked={linked_silver_label_event_count:,}, '
    f'linkage={silver_label_current_device_linkage:.4%}; '
    f'current_devices={label_device_count:,}; '
    f'label_source_max={label_source_max_ts}; cutoff={label_cutoff_date}'
)
failure_label_summary.orderBy('device_category').show(10, truncate=False)


In [ ]:
# ── Cell 6: governed OOS taxonomy and validated-failure allocation ───────────
# Maintenance ledgers are used as a defensive exclusion in addition to the event-level
# commanded flag. A 30-minute tolerance prevents a planned transition from being reported
# as an unplanned hardware OOS onset.

maintenance_exclusion = maintenance.where(
    F.col('maintenance_commanded_oos') | F.col('maintenance_oos')
).select('maintenance_device_id', 'maintenance_start_ts', 'maintenance_end_ts')

maintenance_overlap = (
    events_raw.alias('e')
    .join(
        maintenance_exclusion.alias('m'),
        (F.col('e.device_id') == F.col('m.maintenance_device_id'))
        & (F.col('e.event_ts') >= F.col('m.maintenance_start_ts') - F.expr('INTERVAL 30 MINUTES'))
        & (F.col('e.event_ts') <= F.col('m.maintenance_end_ts') + F.expr('INTERVAL 30 MINUTES')),
        'left',
    )
    .groupBy(F.col('e.event_id').alias('event_id'))
    .agg(
        F.max(
            F.when(F.col('m.maintenance_device_id').isNotNull(), F.lit(1)).otherwise(F.lit(0))
        ).alias('maintenance_overlap_flag')
    )
)

events_evidence = (
    events_raw
    .join(maintenance_overlap, 'event_id', 'left')
    .join(outage, 'event_id', 'left')
    .withColumn('raw_oos_flag', truthy(F.col('is_oos_event')))
    .withColumn('hardware_oos_flag', truthy(F.col('is_hardware_oos_event')))
    .withColumn('commanded_oos_flag', truthy(F.col('is_commanded_oos_event')))
    .withColumn(
        'maintenance_overlap_flag',
        F.coalesce(F.col('maintenance_overlap_flag'), F.lit(0)).cast('boolean'),
    )
    .withColumn(
        'oos_evidence_class',
        F.when(F.col('commanded_oos_flag'), F.lit('COMMANDED_OOS_EXCLUDED'))
        .when(F.col('maintenance_overlap_flag'), F.lit('MAINTENANCE_OOS_EXCLUDED'))
        .when(
            F.col('hardware_oos_flag') & F.col('outage_start').isNotNull(),
            F.lit('HARDWARE_OOS_EPISODE'),
        )
        .when(F.col('hardware_oos_flag'), F.lit('HARDWARE_OOS_EVENT_UNLINKED'))
        .when(F.col('raw_oos_flag'), F.lit('UNCLASSIFIED_OOS_EXCLUDED'))
        .otherwise(F.lit('NON_OOS_EVENT')),
    )
    .withColumn(
        'is_primary_hardware_oos_onset',
        F.col('oos_evidence_class') == 'HARDWARE_OOS_EPISODE',
    )
)

# `device_failures` is device/day grain. Allocate each row to the closest governed onset
# on that device/day, rather than broadcasting it to every event on the day.
failure_candidates = (
    failure_daily.alias('f')
    .join(
        events_evidence.where(F.col('is_primary_hardware_oos_onset')).select(
            'event_id', 'device_id', 'event_date',
            F.coalesce('outage_start', 'event_ts').alias('governed_onset_ts'),
        ).alias('e'),
        (F.col('f.failure_device_id') == F.col('e.device_id'))
        & (F.col('f.failure_date') == F.col('e.event_date')),
        'inner',
    )
    .withColumn(
        '_failure_link_seconds',
        F.abs(F.unix_timestamp('e.governed_onset_ts') - F.unix_timestamp('f.first_failure_dtm')),
    )
    .withColumn(
        '_failure_rank',
        F.row_number().over(
            Window.partitionBy('f.failure_device_id', 'f.failure_date')
            .orderBy(F.col('_failure_link_seconds').asc_nulls_last(), F.col('e.event_id'))
        ),
    )
)

failure_episode_links = (
    failure_candidates
    .where(
        (F.col('_failure_rank') == 1)
        & (
            F.col('_failure_link_seconds').isNull()
            | (F.col('_failure_link_seconds') <= MAX_FAILURE_LINK_HOURS * 3600)
        )
    )
    .select(
        F.col('e.event_id').alias('event_id'),
        F.col('f.failure_device_id').alias('linked_failure_device_id'),
        F.col('f.failure_date').alias('linked_failure_date'),
        F.col('f.validated_failure_events_same_day').alias('validated_failure_events_allocated'),
        F.col('f.first_failure_dtm').alias('allocated_failure_ts'),
        F.col('f.failure_source').alias('allocated_failure_source'),
        F.col('_failure_link_seconds').alias('failure_link_seconds'),
    )
)

failure_linkage_stats = single_row_dict(
    failure_daily.alias('f')
    .join(
        failure_episode_links.alias('l'),
        (F.col('f.failure_device_id') == F.col('l.linked_failure_device_id'))
        & (F.col('f.failure_date') == F.col('l.linked_failure_date')),
        'left',
    )
    .agg(
        F.count('*').alias('validated_failure_device_days'),
        F.sum(F.when(F.col('l.event_id').isNotNull(), 1).otherwise(0)).alias('linked_failure_device_days'),
    )
)

events = (
    events_evidence
    .join(failure_episode_links, 'event_id', 'left')
    .withColumn(
        'failure_evidence_class',
        F.when(
            F.coalesce(F.col('validated_failure_events_allocated'), F.lit(0)) > 0,
            F.lit('VALIDATED_FAILURE_ALLOCATED'),
        )
        .when(
            F.col('is_primary_hardware_oos_onset') & F.col('outage_is_chargeable'),
            F.lit('CHARGEABLE_HARDWARE_OOS'),
        )
        .when(F.col('is_primary_hardware_oos_onset'), F.lit('HARDWARE_OOS_UNCONFIRMED'))
        .otherwise(F.lit('NO_FAILURE_EVIDENCE')),
    )
    .withColumn(
        'is_validated_failure',
        F.col('failure_evidence_class') == 'VALIDATED_FAILURE_ALLOCATED',
    )
    .withColumn(
        'component_serial_id',
        F.when(
            F.trim(F.col('component_serial_nbr').cast('string')) != '',
            F.trim(F.col('component_serial_nbr').cast('string')),
        ),
    )
    .withColumn(
        'outage_episode_id',
        F.coalesce(
            F.col('event_id'),
            F.sha2(F.concat_ws('|', 'device_id', F.col('outage_start').cast('string')), 256),
        ),
    )
)

events = (
    events
    .join(chains, ['device_id', 'event_date'], 'left')
    .join(dim_current, F.col('device_id') == F.col('dim_device_id'), 'left')
    .drop('dim_device_id')
    .withColumn(
        'coordinated_station_flag',
        F.coalesce(F.col('coordinated_station_flag'), F.lit(False)),
    )
    .withColumn('major_station_flag', F.coalesce(F.col('major_station_flag'), F.lit(False)))
    .persist(StorageLevel.DISK_ONLY)
)

label_counts = single_row_dict(events.agg(
    F.count('*').alias('events'),
    F.sum(F.col('raw_oos_flag').cast('long')).alias('raw_oos_events'),
    F.sum(F.col('hardware_oos_flag').cast('long')).alias('hardware_oos_events'),
    F.sum(F.col('is_primary_hardware_oos_onset').cast('long')).alias('hardware_oos_episode_onsets'),
    F.sum(F.col('is_validated_failure').cast('long')).alias('validated_failure_onsets'),
    F.sum(F.coalesce('validated_failure_events_allocated', F.lit(0))).alias('validated_failure_events_allocated'),
    F.sum(
        F.when(
            F.col('is_primary_hardware_oos_onset') & F.col('component_serial_id').isNotNull(), 1,
        ).otherwise(0)
    ).alias('component_serial_populated_oos_onsets'),
))
# The governed event cache is now complete, so the narrower input cache can be released.
events_raw.unpersist(blocking=True)
print('Governed label counts:', label_counts)
print('Validated-failure allocation:', failure_linkage_stats)
print('Spark scratch free after governed-event materialization:', f'{disk_free_gib(SPARK_LOCAL_DIR):.1f} GiB')


In [ ]:
# ── Cell 7: timestamp-accurate cascade construction ──────────────────────────
# A cascade is a contiguous sequence of events for a device.  It is not a daily state
# record. The ordering and real inter-event gaps are retained for all later tests.
w_device_order = Window.partitionBy('device_id').orderBy('event_ts', 'event_id')
events_ordered = (
    events
    .withColumn('_prev_event_ts', F.lag('event_ts').over(w_device_order))
    .withColumn(
        '_new_cascade',
        F.when(F.col('_prev_event_ts').isNull() |
               ((F.unix_timestamp('event_ts') - F.unix_timestamp('_prev_event_ts')) > CASCADE_GAP_MINUTES * 60), 1).otherwise(0)
    )
    .withColumn('_cascade_seq', F.sum('_new_cascade').over(w_device_order.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn('cascade_id', F.concat_ws('|', F.col('device_id').cast('string'), F.lpad(F.col('_cascade_seq').cast('string'), 10, '0')))
    .drop('_prev_event_ts', '_new_cascade')
)

w_cascade = Window.partitionBy('cascade_id').orderBy('event_ts', 'event_id')
w_cascade_all = Window.partitionBy('cascade_id')
events_ordered = (
    events_ordered
    .withColumn('next_subsystem', F.lead('component_subsystem').over(w_cascade))
    .withColumn('next_event_ts', F.lead('event_ts').over(w_cascade))
    .withColumn('edge_lag_seconds', F.unix_timestamp('next_event_ts') - F.unix_timestamp('event_ts'))
    .withColumn('cascade_oos_onset_ts', F.min(F.when(F.col('is_primary_hardware_oos_onset'), F.col('outage_start'))).over(w_cascade_all))
    .withColumn('cascade_has_hardware_oos', F.max(F.col('is_primary_hardware_oos_onset').cast('int')).over(w_cascade_all) == 1)
    .withColumn('cascade_has_validated_failure', F.max(F.col('is_validated_failure').cast('int')).over(w_cascade_all) == 1)
    .withColumn(
        'is_pre_oos_edge',
        F.col('next_subsystem').isNotNull() & F.col('cascade_oos_onset_ts').isNotNull() &
        (F.col('event_ts') < F.col('cascade_oos_onset_ts')) &
        (F.col('next_event_ts') <= F.col('cascade_oos_onset_ts')) &
        (F.col('event_ts') >= F.col('cascade_oos_onset_ts') - F.expr(f'INTERVAL {PRECURSOR_LOOKBACK_HOURS} HOURS'))
    )
)

cascades = (
    events_ordered.groupBy('cascade_id', 'device_id', 'device_category', 'FACILITY_ID', 'FACILITY_NAME', 'BUS_ID')
    .agg(
        F.min('event_ts').alias('cascade_start_ts'), F.max('event_ts').alias('cascade_end_ts'),
        F.count('*').alias('event_count'), F.countDistinct('component_subsystem').alias('distinct_subsystems'),
        F.max(F.col('cascade_has_hardware_oos').cast('int')).cast('boolean').alias('has_hardware_oos'),
        F.max(F.col('cascade_has_validated_failure').cast('int')).cast('boolean').alias('has_validated_failure'),
        F.min('cascade_oos_onset_ts').alias('hardware_oos_onset_ts'),
        F.sum(F.col('outage_duration_min')).alias('hardware_oos_minutes'),
        F.max('coordinated_station_flag').alias('coordinated_station_flag'),
        F.max('major_station_flag').alias('major_station_flag'),
        F.sort_array(F.collect_list(F.struct('event_ts', 'component_subsystem'))).alias('_ordered_subsystems')
    )
    .withColumn('cascade_day', F.to_date('cascade_start_ts'))
    .withColumn('subsystem_sequence', F.concat_ws(' -> ', F.expr('transform(_ordered_subsystems, x -> x.component_subsystem)')))
    .drop('_ordered_subsystems')
)
print('Cascade plan prepared lazily; the reusable edge aggregate is materialized once in Cell 9.')



In [ ]:
# ── Cell 8: Tests 1–4 — quality, labels, trends, and exposure ────────────────
profile_totals = single_row_dict(category_profile.agg(
    F.sum('event_count').alias('all_events'),
    F.sum(F.when(F.col('is_mapped'), F.col('event_count')).otherwise(0)).alias('mapped_events'),
    F.sum(F.when(F.col('is_target_scope'), F.col('event_count')).otherwise(0)).alias('target_events'),
))
category_mapping_coverage = (
    (profile_totals['mapped_events'] or 0) / max(profile_totals['all_events'] or 0, 1)
)
target_scope_share = (
    (profile_totals['target_events'] or 0) / max(profile_totals['all_events'] or 0, 1)
)
target_category_count = events.select('device_category').distinct().count()

outage_linkage = (
    (label_counts['hardware_oos_episode_onsets'] or 0)
    / max(label_counts['hardware_oos_events'] or 0, 1)
)
component_serial_coverage = (
    (label_counts['component_serial_populated_oos_onsets'] or 0)
    / max(label_counts['hardware_oos_episode_onsets'] or 0, 1)
)
duplicate_rate = 1.0 - (post_dedupe_all_n / max(pre_dedupe_n, 1))
validated_failure_linkage = (
    (failure_linkage_stats['linked_failure_device_days'] or 0)
    / max(failure_linkage_stats['validated_failure_device_days'] or 0, 1)
)
label_class_category_count = failure_label_summary.where(
    (F.col('device_category') != 'ALL')
    & (F.col('positive_device_days') > 0)
    & (F.col('negative_device_days') > 0)
).count()

quality_records = [
    ('source_freshness', source_lag_days <= MAX_SOURCE_LAG_DAYS, source_lag_days, f'<= {MAX_SOURCE_LAG_DAYS} days', 'critical' if PRODUCTION_MODE else 'warning', 'Target-family event source lag from run time.'),
    ('direct_label_source_alignment', direct_label_source_lag_days <= 1.0, direct_label_source_lag_days, '<= 1 day behind governed event source', 'warning', 'Direct-S3 Silver label source lag versus the governed event source boundary.'),
    ('event_rows_nonzero', post_dedupe_n > 0, float(post_dedupe_n), '> 0', 'critical', 'Deduplicated TVM/GATE/VALIDATOR event rows.'),
    ('dedupe_rate', duplicate_rate <= 0.01, duplicate_rate, '<= 0.01', 'critical', 'Duplicate event_id share before family scoping.'),
    ('category_mapping_coverage', category_mapping_coverage >= MIN_CATEGORY_MAPPING_COVERAGE, category_mapping_coverage, f'>= {MIN_CATEGORY_MAPPING_COVERAGE}', 'critical', 'Nonblank category-value coverage after canonical normalization; separate from target-scope share.'),
    ('target_scope_share', True, target_scope_share, 'informational', 'info', 'Share of all event rows belonging to TVM/GATE/VALIDATOR; out-of-scope rows are valid.'),
    ('target_category_presence', target_category_count == len(TARGET_DEVICE_CATEGORIES), float(target_category_count), f'= {len(TARGET_DEVICE_CATEGORIES)}', 'critical', 'Expected TVM, GATE, and VALIDATOR families present.'),
    ('hardware_oos_events_nonzero', (label_counts['hardware_oos_events'] or 0) > 0, float(label_counts['hardware_oos_events'] or 0), '> 0', 'critical', 'Silver hardware-OOS event evidence.'),
    ('hardware_oos_episode_linkage', outage_linkage >= 0.50, outage_linkage, '>= 0.50', 'critical', 'Hardware-OOS events linked to governed outage episodes.'),
    ('component_serial_coverage_on_oos', component_serial_coverage >= MIN_COMPONENT_SERIAL_COVERAGE, component_serial_coverage, f'>= {MIN_COMPONENT_SERIAL_COVERAGE}', 'warning', 'Component serial coverage on governed OOS onsets only.'),
    ('validated_failure_allocation', validated_failure_linkage >= 0.50, validated_failure_linkage, '>= 0.50', 'warning', 'Validated device/failure-days allocated to nearest governed onset.'),
    ('direct_silver_label_events_nonzero', linked_silver_label_event_count > 0, float(linked_silver_label_event_count), '> 0', 'critical', 'PS1-compatible hardware-OOS Set events from direct S3 Silver.'),
    ('direct_silver_current_device_linkage', silver_label_current_device_linkage >= 0.95, silver_label_current_device_linkage, '>= 0.95', 'warning', 'PS1 current-device filter retention.'),
    ('label_device_days_nonzero', (label_counts_direct['eligible_device_days'] or 0) > 0, float(label_counts_direct['eligible_device_days'] or 0), '> 0', 'critical', 'Right-censor-safe PS1 daily label population.'),
    ('label_target_category_presence', (label_counts_direct['label_device_categories'] or 0) == len(TARGET_DEVICE_CATEGORIES), float(label_counts_direct['label_device_categories'] or 0), f'= {len(TARGET_DEVICE_CATEGORIES)}', 'critical', 'Label population contains all target families.'),
    ('label_has_both_classes_by_category', label_class_category_count == len(TARGET_DEVICE_CATEGORIES), float(label_class_category_count), f'= {len(TARGET_DEVICE_CATEGORIES)}', 'warning', 'Every family has positive and negative eligible device-days.'),
]
quality_schema = 'check_name string, passed boolean, observed_value double, threshold string, severity string, metric_context string'
quality_df = spark.createDataFrame(quality_records, quality_schema)

critical_failures = [
    row['check_name'] for row in
    quality_df.where((F.col('severity') == 'critical') & (~F.col('passed')))
    .select('check_name').collect()
]
warning_failures = [
    row['check_name'] for row in
    quality_df.where((F.col('severity') == 'warning') & (~F.col('passed')))
    .select('check_name').collect()
]
QUALITY_STATUS = 'PASS' if not critical_failures else 'BLOCKED'
RUN_DISPOSITION = (
    f'{RUN_MODE}_BLOCKED' if QUALITY_STATUS == 'BLOCKED'
    else f'{RUN_MODE}_PASS_WITH_WARNINGS' if warning_failures
    else f'{RUN_MODE}_PASS'
)
quality_df = (
    quality_df
    .withColumn('quality_status', F.lit(QUALITY_STATUS))
    .withColumn('run_mode', F.lit(RUN_MODE))
    .withColumn('run_disposition', F.lit(RUN_DISPOSITION))
)
print(
    f'Quality gate: {RUN_DISPOSITION}; critical_failures={critical_failures or "none"}; '
    f'warning_failures={warning_failures or "none"}'
)
quality_df.orderBy(F.col('passed').asc(), F.col('severity'), F.col('check_name')).show(100, truncate=False)

if QUALITY_STATUS == 'BLOCKED':
    print(
        f'{RUN_MODE} run is BLOCKED by: {", ".join(critical_failures)}. Analytics will '
        'complete so Cell 13 can publish quality and label diagnostics before any optional '
        'production fail-fast status is returned.'
    )
elif not PRODUCTION_MODE:
    print(
        f'{RUN_DISPOSITION}: analytics and charts use isolated prefix '
        f's3://{ARTIFACT_BUCKET}/{EXPORT_PREFIX}; production dispatcher prefix protected.'
    )

# ── V2.5.4 ADDITION: wall-clock device downtime, not a sum of episodes ───────
#
# WHAT WAS WRONG. PS2 computed hardware_oos_minutes as sum(outage_duration_min)
# across episodes. A device is routinely in several concurrent OOS episodes at
# once -- which is what this notebook's own cascade thesis predicts -- so that
# sum counts the same wall-clock minute once per concurrent episode. Measured on
# the 210-day window by the overlap diagnostic:
#
#   fleet      published   correct   overstated   of the sum, double counted
#   GATE         14.66%     3.17%      4.63x                78.4%
#   TVM          30.52%    12.04%      2.53x                60.5%
#   VALIDATOR    95.47%    46.48%      2.05x                51.3%
#
# The published validator figure implied 22.9 out-of-service hours per device
# per day, which a 24-hour day cannot contain.
#
# WHAT THIS BLOCK DOES. It merges each device's OOS intervals and splits the
# merged blocks across calendar days, so hardware_oos_union_minutes is the time
# the DEVICE was out of service. Intervals are built as
# [outage_start, outage_start + outage_duration_min] rather than from
# outage_end, because outage_end carries year-3000 sentinels in this source
# (20,991 rows) while duration_min is the same quantity already clipped at seven
# days. Blocks are bounded by the analysis window, so the per-day explode cannot
# run away.
#
# WHAT IT DOES NOT DO. hardware_oos_minutes is kept unchanged. Summing episode
# durations is still the right measure of COMPONENT burden -- two components
# failing together did cost two component-hours even though the device was down
# once. Only availability needs the union. Both are published, with their ratio,
# so the difference is visible rather than assumed.
#
# ALSO: hardware_oos_onsets counts distinct outage_episode_id, which is derived
# from the source event id. The diagnostic found near-duplicate outage records
# that differ only by that id -- 55.4% of TVM rows, and 98.7% of duplicated GATE
# intervals indistinguishable on subsystem, serial and failure level. So a
# de-duplicated onset count is published beside it, keyed on the interval rather
# than the id. The raw count is retained; neither is silently replaced.
_oos_intervals = (
    events.where(
        F.col('is_primary_hardware_oos_onset')
        & F.col('outage_start').isNotNull()
        & F.col('outage_duration_min').isNotNull()
        & (F.col('outage_duration_min') >= 0)
    )
    .select(
        'device_id',
        F.col('outage_start').cast('timestamp').alias('_iv_start'),
        F.to_timestamp(F.from_unixtime(
            F.col('outage_start').cast('long')
            + (F.col('outage_duration_min') * 60.0).cast('long')
        )).alias('_iv_end'),
    )
    .where((F.col('_iv_start') >= F.lit(analysis_start_ts)) & (F.col('_iv_start') <= F.lit(as_of_ts)))
    .withColumn('_iv_end', F.least(F.greatest(F.col('_iv_end'), F.col('_iv_start')), F.lit(as_of_ts)))
)
_w_iv = Window.partitionBy('device_id').orderBy('_iv_start', '_iv_end')
_oos_blocks = (
    _oos_intervals
    .withColumn('_prior_end', F.max('_iv_end').over(_w_iv.rowsBetween(Window.unboundedPreceding, -1)))
    .withColumn(
        '_new_block',
        F.when(F.col('_prior_end').isNull(), F.lit(1))
         .when(F.col('_iv_start') > F.col('_prior_end'), F.lit(1))
         .otherwise(F.lit(0)),
    )
    .withColumn('_block', F.sum('_new_block').over(_w_iv.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .groupBy('device_id', '_block')
    .agg(F.min('_iv_start').alias('_block_start'), F.max('_iv_end').alias('_block_end'))
)
device_day_union = (
    _oos_blocks
    .withColumn('event_date', F.explode(F.sequence(F.to_date('_block_start'), F.to_date('_block_end'))))
    .withColumn('_day_start', F.col('event_date').cast('timestamp'))
    .withColumn('_day_end', F.col('_day_start') + F.expr('INTERVAL 1 DAY'))
    .withColumn('_ov_start', F.greatest('_block_start', '_day_start'))
    .withColumn('_ov_end', F.least('_block_end', '_day_end'))
    .withColumn('_minutes', F.greatest(F.lit(0.0), (F.col('_ov_end').cast('long') - F.col('_ov_start').cast('long')) / 60.0))
    .groupBy('device_id', 'event_date')
    .agg(F.sum('_minutes').alias('hardware_oos_union_minutes'))
    .persist(StorageLevel.DISK_ONLY)
)
_union_rows = device_day_union.count()
# A device cannot be out of service for more than 1440 minutes in a day. If this
# is not zero the merge is wrong and nothing downstream should be trusted.
_ceiling_breach = device_day_union.where(F.col('hardware_oos_union_minutes') > 1440.5).count()
if _ceiling_breach:
    raise RuntimeError(
        f'{_ceiling_breach:,} device-days exceed 1440 union minutes. The interval merge is '
        'incorrect; refusing to publish an availability measure built on it.'
    )
device_day_onset_dedup = (
    events.where(F.col('is_primary_hardware_oos_onset'))
    .groupBy('device_id', 'event_date')
    .agg(F.countDistinct(F.struct('outage_start', 'outage_duration_min'))
          .alias('hardware_oos_onsets_distinct_interval'))
)
_device_entity = events.select('device_id', 'device_category').dropDuplicates()
fleet_day_union = (
    device_day_union.join(F.broadcast(_device_entity), 'device_id', 'left')
    .groupBy('event_date', 'device_category')
    .agg(F.sum('hardware_oos_union_minutes').alias('hardware_oos_union_minutes'))
)
print(
    f'device-day union: {_union_rows:,} device-days, ceiling breaches={_ceiling_breach} '
    '(must be 0); hardware_oos_minutes retained unchanged for component burden.'
)

oos_governance = (
    events.groupBy('device_category', 'oos_evidence_class', 'failure_evidence_class')
    .agg(
        F.count('*').alias('event_count'),
        F.countDistinct('device_id').alias('device_count'),
        F.sum(F.coalesce('outage_duration_min', F.lit(0.0))).alias('outage_minutes'),
    )
)

daily_oos_trend = (
    events.where(F.col('is_primary_hardware_oos_onset'))
    .groupBy('event_date', 'device_category')
    .agg(
        F.countDistinct('outage_episode_id').alias('hardware_oos_onsets'),
        F.countDistinct(F.struct('device_id', 'outage_start', 'outage_duration_min'))
         .alias('hardware_oos_onsets_distinct_interval'),
        F.countDistinct('device_id').alias('affected_devices'),
        F.sum(F.coalesce('outage_duration_min', F.lit(0.0))).alias('hardware_oos_minutes'),
        F.sum(F.col('is_validated_failure').cast('long')).alias('validated_failure_onsets'),
        F.sum(F.col('outage_is_chargeable').cast('long')).alias('chargeable_oos_onsets'),
    )
)

# Compare PS1's direct-Silver event/day definition with PS2's stricter governed episode/day
# definition. Neither side is silently promoted to the other.
silver_definition_days = (
    silver_hardware_oos_set_events
    .select('device_id', 'device_category', F.col('failure_date').alias('definition_date'))
    .dropDuplicates()
    .withColumn('has_silver_ps1_failure_day', F.lit(1))
)
governed_definition_days = (
    events.where(F.col('is_primary_hardware_oos_onset'))
    .select('device_id', 'device_category', F.col('event_date').alias('definition_date'))
    .dropDuplicates()
    .withColumn('has_governed_oos_episode_day', F.lit(1))
)
definition_day_alignment = (
    silver_definition_days.join(
        governed_definition_days,
        ['device_id', 'device_category', 'definition_date'],
        'full',
    )
    .fillna(0, subset=['has_silver_ps1_failure_day', 'has_governed_oos_episode_day'])
)

def _definition_summary(frame, category_value=None):
    scoped = frame if category_value is None else frame.where(F.col('device_category') == category_value)
    output_category = 'ALL' if category_value is None else category_value
    return scoped.groupBy().agg(
        F.sum('has_silver_ps1_failure_day').alias('silver_ps1_failure_device_days'),
        F.sum('has_governed_oos_episode_day').alias('governed_oos_episode_device_days'),
        F.sum(F.when(
            (F.col('has_silver_ps1_failure_day') == 1)
            & (F.col('has_governed_oos_episode_day') == 1), 1,
        ).otherwise(0)).alias('overlap_device_days'),
        F.sum(F.when(
            (F.col('has_silver_ps1_failure_day') == 1)
            & (F.col('has_governed_oos_episode_day') == 0), 1,
        ).otherwise(0)).alias('silver_only_device_days'),
        F.sum(F.when(
            (F.col('has_silver_ps1_failure_day') == 0)
            & (F.col('has_governed_oos_episode_day') == 1), 1,
        ).otherwise(0)).alias('governed_only_device_days'),
    ).withColumn('device_category', F.lit(output_category))


failure_definition_alignment = _definition_summary(definition_day_alignment)
for _category in TARGET_DEVICE_CATEGORIES:
    failure_definition_alignment = failure_definition_alignment.unionByName(
        _definition_summary(definition_day_alignment, _category)
    )
failure_definition_alignment = (
    failure_definition_alignment
    .withColumn(
        'silver_to_governed_overlap_rate',
        F.col('overlap_device_days') / F.greatest(F.col('silver_ps1_failure_device_days'), F.lit(1)),
    )
    .withColumn(
        'governed_to_silver_overlap_rate',
        F.col('overlap_device_days') / F.greatest(F.col('governed_oos_episode_device_days'), F.lit(1)),
    )
    .withColumn(
        'definition_jaccard',
        F.col('overlap_device_days') / F.greatest(
            F.col('silver_ps1_failure_device_days')
            + F.col('governed_oos_episode_device_days')
            - F.col('overlap_device_days'),
            F.lit(1),
        ),
    )
)

# Exposure, not an invented lost-revenue calculation. Only the affected device/day's own
# PS1 transactions/revenue are aggregated.
ps1_for_impact = SOURCE_FRAMES['ps1']
tap_expr = F.coalesce(F.col('tap_count').cast('double'), F.lit(0.0)) if 'tap_count' in ps1_for_impact.columns else F.lit(0.0)
revenue_expr = F.coalesce(F.col('daily_revenue_cents').cast('double'), F.lit(0.0)) if 'daily_revenue_cents' in ps1_for_impact.columns else F.lit(0.0)
ps1_device_day = ps1_for_impact.select(
    F.col('DEVICE_ID').cast('string').alias('ps1_device_id'),
    F.to_date('transit_day').alias('ps1_event_date'),
    tap_expr.alias('tap_count'),
    revenue_expr.alias('daily_revenue_cents'),
)
daily_oos_trend = (
    daily_oos_trend
    .join(fleet_day_union, ['event_date', 'device_category'], 'left')
    .fillna(0.0, subset=['hardware_oos_union_minutes'])
    .withColumn('oos_minutes_overlap_factor',
                F.when(F.col('hardware_oos_union_minutes') > 0,
                       F.col('hardware_oos_minutes') / F.col('hardware_oos_union_minutes')))
)

affected_device_day = (
    events.where(F.col('is_primary_hardware_oos_onset'))
    .groupBy('device_id', 'event_date', 'device_category')
    .agg(
        F.countDistinct('outage_episode_id').alias('hardware_oos_onsets'),
        F.sum(F.coalesce('outage_duration_min', F.lit(0.0))).alias('hardware_oos_minutes'),
    )
    # V2.5.4: wall-clock device downtime beside the episode sum.
    .join(device_day_union, ['device_id', 'event_date'], 'left')
    .fillna(0.0, subset=['hardware_oos_union_minutes'])
)
impact_exposure = (
    affected_device_day.alias('o')
    .join(
        ps1_device_day.alias('p'),
        (F.col('o.device_id') == F.col('p.ps1_device_id'))
        & (F.col('o.event_date') == F.col('p.ps1_event_date')),
        'left',
    )
    .groupBy(F.col('o.event_date').alias('event_date'), F.col('o.device_category').alias('device_category'))
    .agg(
        F.sum('o.hardware_oos_onsets').alias('hardware_oos_onsets'),
        F.sum('o.hardware_oos_minutes').alias('hardware_oos_minutes'),
        F.sum(F.coalesce(F.col('p.tap_count'), F.lit(0.0))).alias('transactions_exposed'),
        F.sum(F.coalesce(F.col('p.daily_revenue_cents'), F.lit(0.0))).alias('revenue_cents_exposed'),
    )
)


In [ ]:
# ── Cell 9: Tests 5–7 — precursor edges, exact timing, topology ───────────────
def wilson_lower(success, total, z=1.96):
    p = success / F.greatest(total, F.lit(1.0))
    denom = 1.0 + (z * z) / F.greatest(total, F.lit(1.0))
    centre = p + (z * z) / (2.0 * F.greatest(total, F.lit(1.0)))
    adjustment = z * F.sqrt(
        (p * (1.0 - p) + (z * z) / (4.0 * F.greatest(total, F.lit(1.0))))
        / F.greatest(total, F.lit(1.0))
    )
    return F.when(total > 0, (centre - adjustment) / denom).otherwise(F.lit(0.0))


edge_base = events_ordered.where(
    F.col('next_subsystem').isNotNull() & (F.col('edge_lag_seconds') >= 0)
)

# This is the only materialization of the timestamp-ordered edge lineage. Previous
# versions independently recomputed it for precursor, key checks, collection, topology,
# and drift. The compact pair table retains every metric those outputs need.
edge_pair_stats = (
    edge_base
    .groupBy('device_category', 'component_subsystem', 'next_subsystem')
    .agg(
        F.count('*').alias('edge_support'),
        F.sum(F.coalesce(F.col('is_pre_oos_edge').cast('long'), F.lit(0))).alias('pre_oos_edge_count'),
        F.sum(F.coalesce(F.col('cascade_has_validated_failure').cast('long'), F.lit(0))).alias('validated_failure_edge_count'),
        F.expr('percentile_approx(edge_lag_seconds, 0.50)').alias('median_edge_lag_seconds'),
        F.expr('percentile_approx(edge_lag_seconds, 0.95)').alias('p95_edge_lag_seconds'),
        F.sum(F.when(F.col('event_ts') < F.lit(recent_start_ts), 1).otherwise(0)).alias('baseline_edge_count'),
        F.sum(F.when(F.col('event_ts') >= F.lit(recent_start_ts), 1).otherwise(0)).alias('recent_edge_count'),
        F.sum(F.when(
            (F.col('event_ts') < F.lit(recent_start_ts)) & F.coalesce(F.col('is_pre_oos_edge'), F.lit(False)),
            1,
        ).otherwise(0)).alias('baseline_pre_oos_count'),
        F.sum(F.when(
            (F.col('event_ts') >= F.lit(recent_start_ts)) & F.coalesce(F.col('is_pre_oos_edge'), F.lit(False)),
            1,
        ).otherwise(0)).alias('recent_pre_oos_count'),
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
edge_pair_count = edge_pair_stats.count()
print(
    f'Materialized {edge_pair_count:,} subsystem edge pairs; '
    f'Spark scratch free={disk_free_gib(SPARK_LOCAL_DIR):.1f} GiB'
)

edge_baseline = (
    edge_pair_stats.groupBy('device_category')
    .agg(
        (
            F.sum('pre_oos_edge_count')
            / F.greatest(F.sum('edge_support'), F.lit(1))
        ).alias('baseline_pre_oos_rate')
    )
)

precursor_patterns = (
    edge_pair_stats
    .join(edge_baseline, 'device_category')
    .withColumn('pre_oos_rate', F.col('pre_oos_edge_count') / F.greatest(F.col('edge_support'), F.lit(1)))
    .withColumn('pre_oos_wilson_lower_95', wilson_lower(F.col('pre_oos_edge_count').cast('double'), F.col('edge_support').cast('double')))
    .withColumn('pre_oos_lift_vs_category', F.col('pre_oos_rate') / F.greatest(F.col('baseline_pre_oos_rate'), F.lit(1e-9)))
    .withColumn('validated_failure_rate', F.col('validated_failure_edge_count') / F.greatest(F.col('edge_support'), F.lit(1)))
    .withColumn('pattern_key', F.concat_ws(' -> ', 'component_subsystem', 'next_subsystem'))
    .where(F.col('edge_support') >= MIN_PATTERN_SUPPORT)
    .withColumn(
        'evidence_tier',
        F.when(
            (F.col('validated_failure_edge_count') > 0)
            & (F.col('pre_oos_wilson_lower_95') >= 0.05),
            'CONFIRMED_PATTERN',
        )
        .when(F.col('pre_oos_wilson_lower_95') >= 0.05, 'HARDWARE_OOS_PATTERN')
        .otherwise('EXPLORATORY_PATTERN'),
    )
    .withColumn(
        'priority_score',
        F.round(
            F.log1p('edge_support')
            * F.col('pre_oos_lift_vs_category')
            * F.col('pre_oos_wilson_lower_95')
            * (1.0 + F.col('validated_failure_rate')),
            6,
        ),
    )
)

leadlag_timing = precursor_patterns.select(
    'device_category', 'component_subsystem', 'next_subsystem', 'pattern_key', 'edge_support',
    'median_edge_lag_seconds', 'p95_edge_lag_seconds', 'pre_oos_rate',
    'pre_oos_lift_vs_category', 'evidence_tier',
)

out_nodes = (
    edge_pair_stats
    .groupBy('device_category', F.col('component_subsystem').alias('subsystem'))
    .agg(
        F.sum('edge_support').alias('outgoing_edge_volume'),
        F.count('*').alias('out_degree'),
        F.sum('pre_oos_edge_count').alias('_outgoing_pre_oos_count'),
    )
    .withColumn(
        'outgoing_pre_oos_rate',
        F.col('_outgoing_pre_oos_count') / F.greatest(F.col('outgoing_edge_volume'), F.lit(1)),
    )
    .drop('_outgoing_pre_oos_count')
)
in_nodes = (
    edge_pair_stats
    .groupBy('device_category', F.col('next_subsystem').alias('subsystem'))
    .agg(
        F.sum('edge_support').alias('incoming_edge_volume'),
        F.count('*').alias('in_degree'),
    )
)
topology_nodes = (
    out_nodes.join(in_nodes, ['device_category', 'subsystem'], 'full')
    .fillna(
        0,
        subset=[
            'outgoing_edge_volume', 'incoming_edge_volume', 'out_degree', 'in_degree',
            'outgoing_pre_oos_rate',
        ],
    )
    .withColumn(
        'flow_centrality_score',
        F.round(
            F.log1p('outgoing_edge_volume')
            * F.log1p('incoming_edge_volume')
            * (1 + F.col('outgoing_pre_oos_rate')),
            6,
        ),
    )
)


In [ ]:
# ── Cell 10: Tests 8–10 — component serials, trend shifts, clusters ────────────
component_serial_patterns = (
    events.where(F.col('is_primary_hardware_oos_onset') & F.col('component_serial_id').isNotNull())
    .groupBy('device_category', 'component_subsystem', 'component_serial_id')
    .agg(
        F.countDistinct('outage_episode_id').alias('hardware_oos_episode_count'),
        F.sum(F.col('is_validated_failure').cast('long')).alias('validated_failure_count'),
        F.sum(F.coalesce('outage_duration_min', F.lit(0.0))).alias('hardware_oos_minutes'),
        F.countDistinct('event_date').alias('observed_oos_days'),
        F.max('event_ts').alias('last_oos_ts')
    )
    .withColumn('serial_evidence_tier', F.when(F.col('validated_failure_count') > 0, 'VALIDATED_FAILURE_COMPONENT_SERIAL').otherwise('HARDWARE_OOS_COMPONENT_SERIAL'))
    .withColumn('component_priority_score', F.round(F.log1p('hardware_oos_episode_count') * (1 + F.log1p('hardware_oos_minutes')) * (1 + F.col('validated_failure_count')), 6))
    .join(hardware_current, F.col('component_serial_id') == F.col('hardware_component_serial_id'), 'left')
    .drop('hardware_component_serial_id')
    .withColumn('hardware_age_enrichment', F.when(F.col('current_component_age_days').isNotNull(), F.lit('CURRENT_CONFIG_AS_OF_RUN')).otherwise(F.lit('UNAVAILABLE')))
)

# A true date spine makes the rolling-baseline alert denominator explicit. Missing days are
# zeros rather than silently disappearing from a device's own baseline.
entity = events.select('device_id', 'device_category').dropDuplicates()
calendar = spark.createDataFrame([(analysis_start_ts.date(), as_of_ts.date())], 'start date, finish date').select(F.explode(F.sequence('start', 'finish')).alias('event_date'))
device_day_spine = entity.crossJoin(F.broadcast(calendar))
daily_device_oos = events.where(F.col('is_primary_hardware_oos_onset')).groupBy('device_id', 'event_date').agg(
    F.countDistinct('outage_episode_id').alias('hardware_oos_onsets'), F.sum(F.coalesce('outage_duration_min', F.lit(0.0))).alias('hardware_oos_minutes'),
    F.sum(F.col('is_validated_failure').cast('long')).alias('validated_failure_onsets')
)
# V2.5.4: the union is the device's real downtime; the episode sum is component
# burden. The de-duplicated onset count is keyed on the interval rather than the
# source event id, which the diagnostic showed is duplicated.
daily_device_oos = (
    daily_device_oos
    .join(device_day_union, ['device_id', 'event_date'], 'left')
    .join(device_day_onset_dedup, ['device_id', 'event_date'], 'left')
)
device_day = device_day_spine.join(daily_device_oos, ['device_id', 'event_date'], 'left').fillna(
    0, subset=['hardware_oos_onsets', 'hardware_oos_minutes', 'validated_failure_onsets',
               'hardware_oos_union_minutes', 'hardware_oos_onsets_distinct_interval'])
w_daily = Window.partitionBy('device_id').orderBy('event_date').rowsBetween(-28, -1)
device_deterioration = (
    device_day
    .withColumn('baseline_mean_28d', F.avg('hardware_oos_onsets').over(w_daily))
    .withColumn('baseline_std_28d', F.stddev_samp('hardware_oos_onsets').over(w_daily))
    .withColumn('oos_zscore_28d', F.when(F.col('baseline_std_28d') > 0, (F.col('hardware_oos_onsets') - F.col('baseline_mean_28d')) / F.col('baseline_std_28d')))
    .withColumn('alert_reason', F.when((F.col('event_date') >= F.to_date(F.lit(recent_start_ts))) & (F.col('oos_zscore_28d') >= 3.0), 'RECENT_HARDWARE_OOS_SPIKE'))
    .where(F.col('alert_reason').isNotNull())
    .withColumn('oos_minutes_overlap_factor',
                F.when(F.col('hardware_oos_union_minutes') > 0,
                       F.col('hardware_oos_minutes') / F.col('hardware_oos_union_minutes')))
    .select('device_id', 'device_category', 'event_date', 'hardware_oos_onsets',
            'hardware_oos_onsets_distinct_interval', 'hardware_oos_minutes',
            'hardware_oos_union_minutes', 'oos_minutes_overlap_factor',
            'validated_failure_onsets', 'baseline_mean_28d', 'baseline_std_28d',
            'oos_zscore_28d', 'alert_reason')
)

# The grouping is family-aware: gate arrays, validator buses, and facility clusters are
# separate outcomes. A station-wide flag is retained so a common-cause event is not sold as
# a component serial defect.
episode_daily = (
    events.where(F.col('is_primary_hardware_oos_onset'))
    .select(
        'event_date', 'device_id', 'device_category', F.col('FACILITY_ID').cast('string').alias('facility_id'),
        F.col('BUS_ID').cast('string').alias('bus_id'), 'transit_array_id', 'outage_episode_id',
        'coordinated_station_flag', 'major_station_flag'
    )
    .dropDuplicates()
)
array_exposure = (
    dim_current.where((F.col('is_active_current')) & (F.col('dim_device_category') == 'GATE') & F.col('transit_array_id').isNotNull())
    .groupBy('dim_facility_id', 'transit_array_id')
    .agg(F.countDistinct('dim_device_id').alias('observed_group_devices'))
)
facility_clusters = (
    episode_daily.groupBy('event_date', 'device_category', 'facility_id')
    .agg(F.countDistinct('device_id').alias('cofailing_devices'), F.countDistinct('outage_episode_id').alias('hardware_oos_onsets'),
         F.max('coordinated_station_flag').alias('coordinated_station_flag'), F.max('major_station_flag').alias('major_station_flag'))
    .where(F.col('cofailing_devices') >= 2)
    .withColumn('cluster_scope', F.lit('FACILITY')).withColumn('cluster_id', F.col('facility_id'))
    .withColumn('observed_group_devices', F.lit(None).cast('long')).withColumn('cofailure_share', F.lit(None).cast('double'))
)
bus_clusters = (
    episode_daily.where((F.col('device_category') == 'VALIDATOR') & F.col('bus_id').isNotNull())
    .groupBy('event_date', 'bus_id')
    .agg(F.countDistinct('device_id').alias('cofailing_devices'), F.countDistinct('outage_episode_id').alias('hardware_oos_onsets'))
    .where(F.col('cofailing_devices') >= 2)
    .withColumn('device_category', F.lit('VALIDATOR')).withColumnRenamed('bus_id', 'cluster_id')
    .withColumn('facility_id', F.lit(None).cast('string')).withColumn('coordinated_station_flag', F.lit(False)).withColumn('major_station_flag', F.lit(False))
    .withColumn('cluster_scope', F.lit('BUS')).withColumn('observed_group_devices', F.lit(None).cast('long')).withColumn('cofailure_share', F.lit(None).cast('double'))
)
gate_array_clusters = (
    episode_daily.where((F.col('device_category') == 'GATE') & F.col('transit_array_id').isNotNull())
    .groupBy('event_date', 'facility_id', 'transit_array_id')
    .agg(F.countDistinct('device_id').alias('cofailing_devices'), F.countDistinct('outage_episode_id').alias('hardware_oos_onsets'),
         F.max('coordinated_station_flag').alias('coordinated_station_flag'), F.max('major_station_flag').alias('major_station_flag'))
    .alias('g')
    .join(array_exposure.alias('a'),
          (F.col('g.facility_id') == F.col('a.dim_facility_id')) & (F.col('g.transit_array_id') == F.col('a.transit_array_id')), 'left')
    .select('g.*', F.col('a.observed_group_devices'))
    .where(F.col('cofailing_devices') >= 2)
    .withColumn('device_category', F.lit('GATE')).withColumn('cluster_scope', F.lit('GATE_ARRAY'))
    .withColumn('cluster_id', F.concat_ws('|', 'facility_id', 'transit_array_id'))
    .withColumn('cofailure_share', F.when(F.col('observed_group_devices') > 0, F.col('cofailing_devices') / F.col('observed_group_devices')))
)
cofailure_clusters = (
    facility_clusters.select('event_date', 'cluster_scope', 'cluster_id', 'device_category', 'facility_id', 'cofailing_devices', 'hardware_oos_onsets', 'observed_group_devices', 'cofailure_share', 'coordinated_station_flag', 'major_station_flag')
    .unionByName(bus_clusters.select('event_date', 'cluster_scope', 'cluster_id', 'device_category', 'facility_id', 'cofailing_devices', 'hardware_oos_onsets', 'observed_group_devices', 'cofailure_share', 'coordinated_station_flag', 'major_station_flag'))
    .unionByName(gate_array_clusters.select('event_date', 'cluster_scope', 'cluster_id', 'device_category', 'facility_id', 'cofailing_devices', 'hardware_oos_onsets', 'observed_group_devices', 'cofailure_share', 'coordinated_station_flag', 'major_station_flag'))
)



In [ ]:
# ── Cell 11: Tests 11–13 — repair diagnostics, drift, PS integration ──────────
if ENABLE_REPAIR_EFFECTIVENESS:
    repairs = (
        maintenance.where(~F.col('maintenance_commanded_oos') & ~F.col('maintenance_oos'))
        .withColumn('repair_id', F.sha2(F.concat_ws('|', 'maintenance_device_id', F.col('maintenance_start_ts').cast('string'), 'ledger_type', 'maintenance_component_subsystem'), 256))
        .select('repair_id', 'maintenance_device_id', 'maintenance_date', 'maintenance_component_subsystem', 'ledger_type')
        .dropDuplicates(['repair_id'])
    )
    repair_calendar = repairs.withColumn('event_date', F.explode(F.sequence(F.date_sub('maintenance_date', 30), F.date_add('maintenance_date', 30))))
    component_daily = events.where(F.col('is_primary_hardware_oos_onset')).groupBy(
        F.col('device_id'), F.col('component_subsystem'), F.col('event_date')
    ).agg(F.countDistinct('outage_episode_id').alias('oos_episode_count'))
    repair_effectiveness = (
        repair_calendar.alias('r')
        .join(component_daily.alias('d'),
              (F.col('r.maintenance_device_id') == F.col('d.device_id')) &
              (F.col('r.maintenance_component_subsystem') == F.col('d.component_subsystem')) &
              (F.col('r.event_date') == F.col('d.event_date')), 'left')
        .groupBy('repair_id', 'maintenance_component_subsystem', 'ledger_type', 'maintenance_date')
        .agg(
            F.sum(F.when(F.col('r.event_date') < F.col('maintenance_date'), F.coalesce(F.col('d.oos_episode_count'), F.lit(0))).otherwise(0)).alias('pre_30d_oos_onsets'),
            F.sum(F.when(F.col('r.event_date') > F.col('maintenance_date'), F.coalesce(F.col('d.oos_episode_count'), F.lit(0))).otherwise(0)).alias('post_30d_oos_onsets')
        )
        .withColumn('post_vs_pre_change', (F.col('post_30d_oos_onsets') - F.col('pre_30d_oos_onsets')) / F.greatest(F.col('pre_30d_oos_onsets'), F.lit(1)))
        .withColumn('interpretation_note', F.lit('Observational pre/post comparison; not a causal repair-effect estimate.'))
    )
else:
    repair_effectiveness = spark.createDataFrame([], 'repair_id string, maintenance_component_subsystem string, ledger_type string, maintenance_date date, pre_30d_oos_onsets long, post_30d_oos_onsets long, post_vs_pre_change double, interpretation_note string')

edge_drift = (
    edge_pair_stats.select(
        'device_category', 'component_subsystem', 'next_subsystem',
        F.col('baseline_edge_count').alias('BASELINE_count'),
        F.col('recent_edge_count').alias('RECENT_count'),
        F.col('baseline_pre_oos_count').alias('BASELINE_pre_oos_count'),
        F.col('recent_pre_oos_count').alias('RECENT_pre_oos_count'),
    )
    .withColumn('baseline_pre_oos_rate', F.col('BASELINE_pre_oos_count') / F.greatest(F.col('BASELINE_count'), F.lit(1)))
    .withColumn('recent_pre_oos_rate', F.col('RECENT_pre_oos_count') / F.greatest(F.col('RECENT_count'), F.lit(1)))
    .withColumn('rate_change', F.col('recent_pre_oos_rate') - F.col('baseline_pre_oos_rate'))
    .withColumn('drift_flag', (F.col('RECENT_count') >= MIN_PATTERN_SUPPORT) & (F.abs(F.col('rate_change')) >= 0.05))
)

# Exact PS1 performance evaluation against the rebuilt Silver three-day label.
# Cell 24 of the PS1 notebooks writes component-expanded rows, so predictions are reduced
# back to one device/day before scoring. Missing family paths are diagnostic, not fatal.
from pyspark.ml.evaluation import BinaryClassificationEvaluator


def _prediction_col(raw, requested, required=True):
    mapping = _casefold_column_map(raw.columns, 'PS1 cross-wired predictions')
    actual = mapping.get(requested.casefold())
    if required and actual is None:
        raise ValueError(f'PS1 predictions are missing {requested!r}; observed={raw.columns}')
    return actual


prediction_frames = []
prediction_source_status = {}
if ENABLE_PS1_PREDICTION_EVAL:
    for _category in TARGET_DEVICE_CATEGORIES:
        _path = f'{PS1_PREDICTIONS_ROOT}/{_category.lower()}/'
        try:
            _raw = spark.read.option('recursiveFileLookup', 'true').parquet(spark_storage_path(_path))
            _device_id_col = _prediction_col(_raw, 'DEVICE_ID', required=False)
            _device_key_col = _prediction_col(_raw, 'DEVICE_KEY', required=False)
            _date_col = _prediction_col(_raw, 'transit_day', required=False) or _prediction_col(_raw, 'event_date', required=False)
            _prob_col = _prediction_col(_raw, PS1_PREDICTION_PROB_COL)
            _flag_col = _prediction_col(_raw, PS1_PREDICTION_FLAG_COL, required=False)
            _threshold_col = _prediction_col(_raw, PS1_PREDICTION_THRESHOLD_COL, required=False)
            if _device_id_col is None and _device_key_col is None:
                raise ValueError('PS1 predictions need DEVICE_ID or DEVICE_KEY.')
            if _date_col is None:
                raise ValueError('PS1 predictions need transit_day or event_date.')

            _selected = _raw.select(
                (F.col(_device_id_col).cast('string') if _device_id_col else F.lit(None).cast('string')).alias('_prediction_device_id'),
                (F.col(_device_key_col).cast('string') if _device_key_col else F.lit(None).cast('string')).alias('_prediction_device_key'),
                F.to_date(F.col(_date_col)).alias('score_date'),
                F.col(_prob_col).cast('double').alias('ps1_fail_prob'),
                (F.col(_flag_col).cast('int') if _flag_col else F.lit(None).cast('int')).alias('_ps1_predicted'),
                (F.col(_threshold_col).cast('double') if _threshold_col else F.lit(0.5)).alias('threshold_used'),
            )
            if _device_id_col is None:
                _selected = (
                    _selected.alias('p')
                    .join(
                        label_device_map.select('label_device_key', 'label_device_id').alias('d'),
                        F.col('p._prediction_device_key') == F.col('d.label_device_key'),
                        'left',
                    )
                    .withColumn('_prediction_device_id', F.col('d.label_device_id'))
                    .select('p.*', F.col('d.label_device_id').alias('_mapped_device_id'))
                    .withColumn('_prediction_device_id', F.coalesce('_prediction_device_id', '_mapped_device_id'))
                    .drop('_mapped_device_id')
                )
            _selected = (
                _selected
                .withColumn('device_id', F.col('_prediction_device_id'))
                .withColumn('device_category', F.lit(_category))
                .withColumn(
                    'ps1_predicted',
                    F.coalesce(
                        F.col('_ps1_predicted'),
                        (F.col('ps1_fail_prob') >= F.coalesce('threshold_used', F.lit(0.5))).cast('int'),
                    ),
                )
                .where(F.col('device_id').isNotNull() & F.col('score_date').isNotNull())
                .groupBy('device_id', 'device_category', 'score_date')
                .agg(
                    F.max('ps1_fail_prob').alias('ps1_fail_prob'),
                    F.max('ps1_predicted').alias('ps1_predicted'),
                    F.max('threshold_used').alias('threshold_used'),
                )
            )
            prediction_frames.append(_selected)
            prediction_source_status[_category] = 'AVAILABLE'
        except Exception as _prediction_error:
            prediction_source_status[_category] = f'UNAVAILABLE: {str(_prediction_error)[:240]}'
            print(f'[PS1 evaluation] {_category} skipped at {_path}: {_prediction_error}')
else:
    prediction_source_status = {c: 'DISABLED' for c in TARGET_DEVICE_CATEGORIES}

if prediction_frames:
    ps1_predictions = prediction_frames[0]
    for _frame in prediction_frames[1:]:
        ps1_predictions = ps1_predictions.unionByName(_frame)
    ps1_eval_rows = (
        failure_label_panel.select(
            'device_id', 'device_category', F.col('label_date').alias('score_date'),
            F.col(PS1_LABEL_COL).cast('double').alias('actual_label'),
        )
        .join(ps1_predictions, ['device_id', 'device_category', 'score_date'], 'inner')
        .where(F.col('ps1_fail_prob').between(0.0, 1.0))
        .persist(StorageLevel.DISK_ONLY)
    )
else:
    ps1_eval_rows = spark.createDataFrame(
        [],
        'device_id string, device_category string, score_date date, actual_label double, '
        'ps1_fail_prob double, ps1_predicted int, threshold_used double',
    )

eligible_counts = {
    row['device_category']: int(row['eligible_device_days'])
    for row in failure_label_summary.select('device_category', 'eligible_device_days').collect()
}


def _safe_ratio(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else None


def _evaluate_ps1_scope(category):
    scoped = ps1_eval_rows if category == 'ALL' else ps1_eval_rows.where(F.col('device_category') == category)
    stats = single_row_dict(scoped.agg(
        F.count('*').alias('evaluated_device_days'),
        F.sum(F.when((F.col('actual_label') == 1) & (F.col('ps1_predicted') == 1), 1).otherwise(0)).alias('tp'),
        F.sum(F.when((F.col('actual_label') == 0) & (F.col('ps1_predicted') == 1), 1).otherwise(0)).alias('fp'),
        F.sum(F.when((F.col('actual_label') == 0) & (F.col('ps1_predicted') == 0), 1).otherwise(0)).alias('tn'),
        F.sum(F.when((F.col('actual_label') == 1) & (F.col('ps1_predicted') == 0), 1).otherwise(0)).alias('fn'),
        F.avg(F.pow(F.col('ps1_fail_prob') - F.col('actual_label'), 2)).alias('brier_score'),
        F.avg('actual_label').alias('actual_positive_rate'),
        F.avg(F.col('ps1_predicted').cast('double')).alias('predicted_positive_rate'),
    ))
    n = int(stats['evaluated_device_days'] or 0)
    tp, fp, tn, fn = [int(stats[k] or 0) for k in ('tp', 'fp', 'tn', 'fn')]
    precision = _safe_ratio(tp, tp + fp)
    recall = _safe_ratio(tp, tp + fn)
    specificity = _safe_ratio(tn, tn + fp)
    f1 = _safe_ratio(2 * tp, 2 * tp + fp + fn)
    balanced_accuracy = (
        (recall + specificity) / 2.0
        if recall is not None and specificity is not None
        else None
    )
    roc_auc = pr_auc = None
    if n and (tp + fn) and (tn + fp):
        roc_auc = BinaryClassificationEvaluator(
            labelCol='actual_label', rawPredictionCol='ps1_fail_prob', metricName='areaUnderROC'
        ).evaluate(scoped)
        pr_auc = BinaryClassificationEvaluator(
            labelCol='actual_label', rawPredictionCol='ps1_fail_prob', metricName='areaUnderPR'
        ).evaluate(scoped)
    eligible = int(eligible_counts.get(category, 0))
    return (
        category, n, eligible, _safe_ratio(n, eligible), tp, fp, tn, fn,
        stats['actual_positive_rate'], stats['predicted_positive_rate'], precision, recall,
        specificity, f1, balanced_accuracy, stats['brier_score'], roc_auc, pr_auc,
        'EVALUATED' if n else prediction_source_status.get(category, 'NO_MATCHED_PREDICTIONS'),
    )


performance_rows = [_evaluate_ps1_scope('ALL')]
performance_rows.extend(_evaluate_ps1_scope(c) for c in TARGET_DEVICE_CATEGORIES)
performance_schema = (
    'device_category string, evaluated_device_days long, eligible_device_days long, '
    'prediction_coverage double, true_positive long, false_positive long, true_negative long, '
    'false_negative long, actual_positive_rate double, predicted_positive_rate double, '
    'precision double, recall double, specificity double, f1_score double, '
    'balanced_accuracy double, brier_score double, roc_auc double, pr_auc double, '
    'evaluation_status string'
)
ps1_model_performance = spark.createDataFrame(performance_rows, performance_schema)

# Evaluation results are now local rows; release prediction and device-map caches before
# the large pattern-output phase.
if getattr(ps1_eval_rows, 'is_cached', False):
    ps1_eval_rows.unpersist(blocking=True)
if getattr(label_device_map, 'is_cached', False):
    label_device_map.unpersist(blocking=True)

# PS3 confirmation requires fault/failure evidence. Chargeability alone is not treated as
# hardware-failure confirmation.
ps3_daily = (
    SOURCE_FRAMES['ps3']
    .select(
        F.col('device_id').cast('string').alias('ps3_device_id'),
        F.to_date('transit_day').alias('ps3_event_date'),
        (
            truthy(F.col('is_device_fault'))
            | (F.coalesce(F.col('failure_level').cast('double'), F.lit(0.0)) > 0)
        ).alias('ps3_confirming_evidence'),
    )
    .groupBy('ps3_device_id', 'ps3_event_date')
    .agg(F.max('ps3_confirming_evidence').alias('ps3_confirming_evidence'))
)
governed_onset_days = events.where(F.col('is_primary_hardware_oos_onset')).select(
    'device_id', 'event_date', 'outage_episode_id'
).dropDuplicates()

ps3_alignment = (
    governed_onset_days.alias('e')
    .join(
        ps3_daily.alias('p'),
        (F.col('e.device_id') == F.col('p.ps3_device_id'))
        & (F.col('e.event_date') == F.col('p.ps3_event_date')),
        'left',
    )
    .groupBy().agg(
        F.countDistinct('e.outage_episode_id').alias('truth_positive_count'),
        F.sum(F.coalesce(F.col('p.ps3_confirming_evidence').cast('long'), F.lit(0))).alias('matched_positive_count'),
    )
    .withColumn('signal_positive_count', F.lit(None).cast('long'))
    .withColumn('precision', F.lit(None).cast('double'))
    .withColumn('recall', F.col('matched_positive_count') / F.greatest(F.col('truth_positive_count'), F.lit(1)))
    .withColumn('population_unit', F.lit('GOVERNED_OOS_EPISODE'))
    .withColumn('alignment_status', F.lit('EXACT_PS3_DEVICE_FAULT_OR_FAILURE_LEVEL_SAME_DAY'))
)

ps4_daily = (
    SOURCE_FRAMES['ps4']
    .select(
        F.col('DEVICE_ID').cast('string').alias('device_id'),
        F.to_date('transit_day').alias('event_date'),
        truthy(F.col('ensemble_anomaly_flag')).alias('ps4_anomaly'),
    )
    .groupBy('device_id', 'event_date')
    .agg(F.max('ps4_anomaly').alias('ps4_anomaly'))
)
ps4_alignment = (
    governed_onset_days
    .join(
        ps4_daily.select('device_id', F.date_add('event_date', 1).alias('event_date'), 'ps4_anomaly'),
        ['device_id', 'event_date'],
        'left',
    )
    .groupBy().agg(
        F.countDistinct('outage_episode_id').alias('truth_positive_count'),
        F.sum(F.coalesce(F.col('ps4_anomaly').cast('long'), F.lit(0))).alias('matched_positive_count'),
    )
    .withColumn('signal_positive_count', F.lit(None).cast('long'))
    .withColumn('precision', F.lit(None).cast('double'))
    .withColumn('recall', F.col('matched_positive_count') / F.greatest(F.col('truth_positive_count'), F.lit(1)))
    .withColumn('population_unit', F.lit('GOVERNED_OOS_EPISODE'))
    .withColumn('alignment_status', F.lit('EXACT_PS4_ENSEMBLE_ANOMALY_PREVIOUS_DAY'))
)

ps1_alignment = (
    ps1_model_performance.where(F.col('device_category') == 'ALL')
    .select(
        (F.col('true_positive') + F.col('false_negative')).alias('truth_positive_count'),
        (F.col('true_positive') + F.col('false_positive')).alias('signal_positive_count'),
        F.col('true_positive').alias('matched_positive_count'),
        'precision', 'recall',
        F.lit('ELIGIBLE_DEVICE_DAY').alias('population_unit'),
        F.concat(F.lit('DIRECT_SILVER_3D_LABEL|'), F.col('evaluation_status')).alias('alignment_status'),
    )
)

cross_ps_alignment = (
    ps1_alignment.withColumn('source', F.lit('PS1'))
    .unionByName(ps3_alignment.withColumn('source', F.lit('PS3')))
    .unionByName(ps4_alignment.withColumn('source', F.lit('PS4')))
)

ps1_model_performance.orderBy('device_category').show(10, truncate=False)


In [ ]:
# ── Cell 12: dashboard output writer — one materialization, manifest last ─────
def snake(name):
    import re
    value = re.sub(r'[^a-zA-Z0-9]+', '_', name).strip('_').lower()
    return value[:63]


MANIFESTS = []
PUBLISHED_PANDAS = {}
CHART_TABLES = {
    'ps2_v25_failure_label_daily',
    'ps2_v2_daily_oos_trend',
    'ps2_v2_precursor_patterns',
    'ps2_v2_component_serial_patterns',
}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def ensure_spark_disk_headroom(stage):
    free_gib = disk_free_gib(SPARK_LOCAL_DIR)
    if free_gib < MIN_PUBLISH_FREE_GIB:
        raise RuntimeError(
            f'PS2 stopped before {stage}: Spark scratch has only {free_gib:.1f} GiB free at '
            f'{SPARK_LOCAL_DIR}; required >= {MIN_PUBLISH_FREE_GIB:.1f} GiB. Restart the '
            'kernel with a larger PS2_SPARK_LOCAL_DIR and use Run All. No completion marker '
            'was written for this partial run.'
        )
    return free_gib


def write_dashboard_table(df, table_name, primary_keys, load_mode='upsert'):
    """Bounded collect once, validate the same rows, upload data, then publish manifest."""
    table_name = snake(table_name)
    normalized = df.toDF(*[snake(c) for c in df.columns])
    normalized = (
        normalized
        .withColumn('run_id', F.lit(RUN_ID))
        .withColumn('run_ts_utc', F.lit(RUN_TS.isoformat()).cast('timestamp'))
        .withColumn('as_of_ts', F.lit(as_of_ts).cast('timestamp'))
        .withColumn('source_start_ts', F.lit(analysis_start_ts).cast('timestamp'))
        .withColumn('source_end_ts', F.lit(as_of_ts).cast('timestamp'))
        .withColumn('notebook_version', F.lit(NOTEBOOK_VERSION))
        .withColumn('city_id', F.lit(CITY_ID))
        .withColumn('quality_status', F.lit(QUALITY_STATUS))
        .withColumn('run_mode', F.lit(RUN_MODE))
        .withColumn('run_disposition', F.lit(RUN_DISPOSITION))
        .withColumn('output_scope', F.lit(OUTPUT_SCOPE))
        .withColumn('is_production', F.lit(PRODUCTION_MODE))
    )
    keys = [snake(k) for k in primary_keys]
    missing_keys = sorted(set(keys) - set(normalized.columns))
    if missing_keys:
        raise ValueError(f'{table_name}: declared RDS keys are missing after normalization: {missing_keys}')

    free_before_gib = ensure_spark_disk_headroom(f'collecting {table_name}')
    # One Spark action replaces count + duplicate-group shuffle + toPandas. The extra row
    # proves whether the single-file guardrail was exceeded without scanning the lineage again.
    pdf = normalized.limit(MAX_OUTPUT_ROWS + 1).toPandas()
    n_rows = len(pdf)
    if n_rows > MAX_OUTPUT_ROWS:
        raise RuntimeError(
            f'{table_name} exceeds the {MAX_OUTPUT_ROWS:,}-row single-file Loader guardrail. '
            'Aggregate further or upgrade the Loader Lambda to consume partitioned data.'
        )
    if pdf[keys].isna().to_numpy().any():
        null_counts = {key: int(pdf[key].isna().sum()) for key in keys if pdf[key].isna().any()}
        raise RuntimeError(f'{table_name} has null RDS key values: {null_counts}')
    duplicate_mask = pdf.duplicated(subset=keys, keep=False)
    if duplicate_mask.any():
        duplicate_groups = int(pdf.loc[duplicate_mask, keys].drop_duplicates().shape[0])
        examples = pdf.loc[duplicate_mask, keys].drop_duplicates().head(5).to_dict('records')
        raise RuntimeError(
            f'{table_name} violates its declared key contract: {duplicate_groups:,} duplicate '
            f'key groups; examples={examples}'
        )

    local_dir = LOCAL_STAGE / table_name
    local_dir.mkdir(parents=True, exist_ok=True)
    local_parquet = local_dir / 'part-0.parquet'
    local_manifest = local_dir / 'manifest.json'
    pdf.to_parquet(local_parquet, index=False)
    data_sha = sha256_file(local_parquet)
    data_key = f'{EXPORT_PREFIX}/{table_name}/computed_date={as_of_ts.date().isoformat()}/run_id={RUN_ID}/part-0.parquet'
    manifest_key = f'{EXPORT_PREFIX}/{table_name}/computed_date={as_of_ts.date().isoformat()}/run_id={RUN_ID}/manifest.json'
    s3.upload_file(str(local_parquet), ARTIFACT_BUCKET, data_key)
    manifest = {
        'manifest_version': '2.1', 'pipeline': 'ps2', 'table_name': table_name,
        'row_count': int(n_rows), 'columns': list(pdf.columns), 'primary_keys': keys,
        'load_mode': load_mode, 'idempotency_key': f'{CITY_ID}|{table_name}|{as_of_ts.date().isoformat()}|{RUN_ID}',
        'run_id': RUN_ID, 'computed_at_utc': RUN_TS.isoformat(), 'as_of_ts': str(as_of_ts),
        'source_window': {'start_ts': str(analysis_start_ts), 'end_ts': str(as_of_ts)},
        'notebook_version': NOTEBOOK_VERSION, 'quality_status': QUALITY_STATUS,
        'run_mode': RUN_MODE, 'run_disposition': RUN_DISPOSITION,
        'output_scope': OUTPUT_SCOPE, 'is_production': PRODUCTION_MODE,
        'label_contract': LABEL_CONTRACT,
        'runtime': {
            'profile': RUNTIME_PROFILE['name'], 'python': RUNTIME_PROFILE['python'],
            'spark': spark.version, 'delta': RUNTIME_PROFILE['delta_spark'],
            'hadoop': HADOOP_VERSION, 'java': JAVA_VERSION_LINE,
            'master': SPARK_MASTER, 'driver_memory': SPARK_DRIVER_MEMORY,
            'shuffle_partitions': SPARK_SHUFFLE_PARTITIONS,
            'effective_memory_gib': round(EFFECTIVE_MEMORY_GIB, 1),
            'spark_local_dir': str(SPARK_LOCAL_DIR),
            'spark_free_gib_before_collect': round(free_before_gib, 2),
            'spark_free_gib_after_collect': round(disk_free_gib(SPARK_LOCAL_DIR), 2),
        },
        's3_data_key': data_key, 'sha256': data_sha,
    }
    local_manifest.write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
    s3.upload_file(str(local_manifest), ARTIFACT_BUCKET, manifest_key)  # manifest is deliberately last
    MANIFESTS.append({**manifest, 's3_manifest_key': manifest_key})
    if table_name in CHART_TABLES:
        PUBLISHED_PANDAS[table_name] = pdf
    print(f'Published {table_name}: {n_rows:,} rows -> s3://{ARTIFACT_BUCKET}/{manifest_key}')

    # Successful uploads no longer need their local serialization copies.
    local_parquet.unlink(missing_ok=True)
    local_manifest.unlink(missing_ok=True)
    try:
        local_dir.rmdir()
    except OSError:
        pass
    return pdf


In [ ]:
# ── Cell 13: Tests 14–15 — phased publish and visual assets ──────────────────
PUBLISH_STATE_REQUIRED = [
    'QUALITY_STATUS', 'quality_df', 'category_profile', 'failure_label_summary',
    'failure_label_daily', 'failure_horizon_profile', 'ps1_label_parity',
    'failure_definition_alignment', 'ps1_model_performance', 'oos_governance',
    'daily_oos_trend', 'impact_exposure', 'precursor_patterns', 'leadlag_timing',
    'topology_nodes', 'component_serial_patterns', 'device_deterioration',
    'cofailure_clusters', 'repair_effectiveness', 'edge_drift', 'cross_ps_alignment',
    'edge_pair_stats', 'events', 'write_dashboard_table', 'MANIFESTS',
    'PUBLISHED_PANDAS', 's3', 'as_of_ts', 'RUNTIME_PROFILE', 'RUN_MODE',
    'RUN_DISPOSITION', 'OUTPUT_SCOPE', 'critical_failures', 'PRODUCTION_MODE',
    'FAIL_FAST_ON_CRITICAL_GATE',
]
missing_publish_state = [name for name in PUBLISH_STATE_REQUIRED if name not in globals()]
if missing_publish_state:
    raise RuntimeError(
        'Cell 13 is the publish step. Start with Cell 0 and run Cells 1–12 in order '
        '(or use Run All) before publishing. Missing state: '
        f'{", ".join(missing_publish_state)}'
    )
if QUALITY_STATUS not in {'PASS', 'BLOCKED'}:
    raise RuntimeError(
        'The governed OOS/data-quality gate has not completed. Run Cell 8 before '
        'publishing; QUALITY_STATUS must be PASS or BLOCKED.'
    )


def release_spark_cache(*names):
    released = []
    for name in names:
        frame = globals().get(name)
        if frame is not None and getattr(frame, 'is_cached', False):
            frame.unpersist(blocking=True)
            released.append(name)
    if released:
        print('Released Spark caches:', ', '.join(released))
    print('Spark scratch free:', f'{disk_free_gib(SPARK_LOCAL_DIR):.1f} GiB')


def upload_chart(fig, filename):
    path = LOCAL_STAGE / filename
    fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    key = f'{EXPORT_PREFIX}/charts/as_of_date={as_of_ts.date().isoformat()}/run_id={RUN_ID}/{filename}'
    s3.upload_file(str(path), ARTIFACT_BUCKET, key)
    path.unlink(missing_ok=True)
    return f's3://{ARTIFACT_BUCKET}/{key}'


def publish_run_completion(status):
    completion = {
        'pipeline': 'ps2', 'run_id': RUN_ID, 'status': status,
        'quality_status': QUALITY_STATUS, 'run_mode': RUN_MODE,
        'run_disposition': RUN_DISPOSITION, 'output_scope': OUTPUT_SCOPE,
        'notebook_version': NOTEBOOK_VERSION, 'as_of_ts': str(as_of_ts),
        'completed_at_utc': dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
        'manifest_count': len(MANIFESTS),
        'manifest_keys': [item['s3_manifest_key'] for item in MANIFESTS],
    }
    path = LOCAL_STAGE / 'run_complete.json'
    path.write_text(json.dumps(completion, indent=2), encoding='utf-8')
    key = f'{EXPORT_PREFIX}/_runs/as_of_date={as_of_ts.date().isoformat()}/run_id={RUN_ID}/run_complete.json'
    s3.upload_file(str(path), ARTIFACT_BUCKET, key)
    path.unlink(missing_ok=True)
    return f's3://{ARTIFACT_BUCKET}/{key}'


if QUALITY_STATUS == 'PASS':
    # Phase 1: governance and PS1-compatible failure-label products.
    write_dashboard_table(quality_df, 'ps2_v25_run_quality', ['check_name'])
    write_dashboard_table(category_profile, 'ps2_v25_category_profile', ['device_category_raw', 'device_category'])
    write_dashboard_table(failure_label_summary, 'ps2_v25_failure_label_summary', ['device_category'])
    write_dashboard_table(failure_label_daily, 'ps2_v25_failure_label_daily', ['label_date', 'device_category'])
    write_dashboard_table(failure_horizon_profile, 'ps2_v25_failure_horizon_profile', ['device_category', 'lead_day'])
    write_dashboard_table(ps1_label_parity, 'ps2_v25_ps1_label_parity', ['device_category'])
    write_dashboard_table(failure_definition_alignment, 'ps2_v25_failure_definition_alignment', ['device_category'])
    write_dashboard_table(ps1_model_performance, 'ps2_v25_ps1_model_performance', ['device_category'])
    release_spark_cache(
        'category_profile', 'failure_label_panel', 'label_base_daily',
        'silver_hardware_oos_set_events',
    )

    # Phase 2: governed OOS, exposure, pattern, component, and cross-PS products.
    write_dashboard_table(oos_governance, 'ps2_v2_oos_governance', ['device_category', 'oos_evidence_class', 'failure_evidence_class'])
    write_dashboard_table(daily_oos_trend, 'ps2_v2_daily_oos_trend', ['event_date', 'device_category'])
    write_dashboard_table(impact_exposure, 'ps2_v2_customer_exposure', ['event_date', 'device_category'])
    write_dashboard_table(precursor_patterns, 'ps2_v2_precursor_patterns', ['device_category', 'component_subsystem', 'next_subsystem'])
    write_dashboard_table(leadlag_timing, 'ps2_v2_leadlag_timing', ['device_category', 'component_subsystem', 'next_subsystem'])
    write_dashboard_table(topology_nodes, 'ps2_v2_topology_nodes', ['device_category', 'subsystem'])
    write_dashboard_table(component_serial_patterns, 'ps2_v2_component_serial_patterns', ['device_category', 'component_subsystem', 'component_serial_id'])
    write_dashboard_table(device_deterioration, 'ps2_v2_device_deterioration', ['device_id', 'event_date', 'alert_reason'])
    write_dashboard_table(cofailure_clusters, 'ps2_v2_cofailure_clusters', ['event_date', 'cluster_scope', 'cluster_id', 'device_category'])
    write_dashboard_table(repair_effectiveness, 'ps2_v2_repair_effectiveness', ['repair_id'])
    write_dashboard_table(edge_drift, 'ps2_v2_pattern_drift', ['device_category', 'component_subsystem', 'next_subsystem'])
    write_dashboard_table(cross_ps_alignment, 'ps2_v2_cross_ps_alignment', ['source'])

    # Charts use the exact rows already validated and uploaded; no Spark lineage is rerun.
    sns.set_theme(style='whitegrid')
    label_pd = PUBLISHED_PANDAS['ps2_v25_failure_label_daily'].sort_values(['label_date', 'device_category'])
    if not label_pd.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.lineplot(data=label_pd, x='label_date', y='label_positive_rate', hue='device_category', ax=ax)
        ax.set_title('PS1-compatible future hardware-OOS label prevalence')
        ax.set_xlabel('Eligible score date'); ax.set_ylabel('Positive device-day rate')
        ax.set_ylim(bottom=0)
        print('Chart:', upload_chart(fig, 'ps1_hardware_oos_3d_label_rate.png'))

    trend_pd = PUBLISHED_PANDAS['ps2_v2_daily_oos_trend'].sort_values(['event_date', 'device_category'])
    if not trend_pd.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.lineplot(data=trend_pd, x='event_date', y='hardware_oos_onsets', hue='device_category', marker='o', ax=ax)
        ax.set_title('Governed hardware-OOS onsets by device family')
        ax.set_xlabel('Event date'); ax.set_ylabel('Hardware-OOS episode onsets')
        print('Chart:', upload_chart(fig, 'hardware_oos_trend.png'))

    pattern_pd = PUBLISHED_PANDAS['ps2_v2_precursor_patterns'].nlargest(15, 'priority_score')
    if not pattern_pd.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        sns.barplot(data=pattern_pd.sort_values('priority_score'), x='priority_score', y='pattern_key', hue='device_category', dodge=False, ax=ax)
        ax.set_title('Top timestamp-validated precursor patterns')
        ax.set_xlabel('Explainable priority score'); ax.set_ylabel('Subsystem transition')
        print('Chart:', upload_chart(fig, 'top_precursor_patterns.png'))

    serial_pd = PUBLISHED_PANDAS['ps2_v2_component_serial_patterns'].nlargest(15, 'component_priority_score')
    if not serial_pd.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        sns.barplot(data=serial_pd.sort_values('component_priority_score'), x='component_priority_score', y='component_serial_id', hue='device_category', dodge=False, ax=ax)
        ax.set_title('Highest-burden component serials (hardware-OOS evidence)')
        ax.set_xlabel('Component priority score'); ax.set_ylabel('Component serial')
        print('Chart:', upload_chart(fig, 'component_serial_priority.png'))

    release_spark_cache('edge_pair_stats', 'events')
    spark.catalog.clearCache()
    RUN_COMPLETION_URI = publish_run_completion('COMPLETE')
else:
    # A blocked run publishes small audit tables only; analytical tables stay protected.
    write_dashboard_table(quality_df, 'ps2_v25_run_quality', ['check_name'], load_mode='append_audit')
    write_dashboard_table(category_profile, 'ps2_v25_category_profile_audit', ['device_category_raw', 'device_category'], load_mode='append_audit')
    write_dashboard_table(failure_label_summary, 'ps2_v25_failure_label_summary_audit', ['device_category'], load_mode='append_audit')
    write_dashboard_table(ps1_label_parity, 'ps2_v25_ps1_label_parity_audit', ['device_category'], load_mode='append_audit')
    write_dashboard_table(failure_definition_alignment, 'ps2_v25_failure_definition_alignment_audit', ['device_category'], load_mode='append_audit')
    write_dashboard_table(ps1_model_performance, 'ps2_v25_ps1_model_performance_audit', ['device_category'], load_mode='append_audit')
    release_spark_cache(
        'category_profile', 'failure_label_panel', 'label_base_daily',
        'silver_hardware_oos_set_events', 'edge_pair_stats', 'events',
    )
    spark.catalog.clearCache()
    RUN_COMPLETION_URI = publish_run_completion('BLOCKED_AUDIT_COMPLETE')
    print('Dashboard publishing blocked; quality/category/label diagnostics were published for review.')

print('\nRun status:', QUALITY_STATUS)
print('Run disposition:', RUN_DISPOSITION)
print('Output scope:', OUTPUT_SCOPE)
print('Output prefix:', f's3://{ARTIFACT_BUCKET}/{EXPORT_PREFIX}')
print('Completion marker:', RUN_COMPLETION_URI)
print('Manifests written:', len(MANIFESTS))
for manifest in MANIFESTS:
    print(f"  {manifest['table_name']}: {manifest['row_count']:,} rows | {manifest['s3_manifest_key']}")

if QUALITY_STATUS == 'BLOCKED' and PRODUCTION_MODE and FAIL_FAST_ON_CRITICAL_GATE:
    raise RuntimeError(
        'PS2 PRODUCTION run blocked by critical governance checks after audit manifests were '
        f'published: {", ".join(critical_failures)}'
    )


## Operations hand-off — v2.5.1

This revision retains the v2.5 PS1-compatible `will_hardware_oos_3d` label and every
governed PS2 output. It changes execution mechanics, not the metric definitions.

### Scheduled-job settings

- `PS2_RUN_MODE=PRODUCTION`
- `PS2_SPARK_LOCAL_DIR=/path/on/a/large/attached/volume/run-id` when automatic selection
  is not appropriate. The default preflight requires 20 GiB free; tune with
  `PS2_MIN_SPARK_FREE_GIB` only after sizing the real workload.
- `PS2_LABEL_EVENTS_PATH`, `PS2_LABEL_DIM_DEVICE_PATH`, and
  `PS2_PS1_PREDICTIONS_ROOT` when the stable defaults differ.

Each dashboard result is executed once, bounded by `PS2_MAX_OUTPUT_ROWS`, validated for
nonnull/unique Loader keys in the collected result, serialized, uploaded, and followed by
its manifest. A final `_runs/.../run_complete.json` marker is written only after every
expected table and chart has succeeded. Runs without that marker are partial and must not
be treated as complete by orchestration or monitoring.

After a `No space left on device` failure, restart the kernel before using Run All so the
failed SparkContext and its block manager are gone. Do not resume at the publish cell.


## Downstream contract - PS2 notebook to dashboard

Non-executable. Records the guarantees this notebook makes to everything
downstream of it, so a later change here does not silently break the chain.
Items 1, 2, 4, 5, 6 and 7 are implemented in the Lambda, the S3 notification
config and the API - not in this notebook. They are written down here because
this notebook is the producer they all depend on.

**What this notebook guarantees**

| Guarantee | Where |
|---|---|
| Partition key is `computed_date=`, matching the loader regex | Cell 12 |
| `run_id` sorts chronologically and parses as a UUID | Cell -1 |
| `manifest.json` is uploaded after its `part-0.parquet` | Cell 12 |
| `run_complete.json` is uploaded after the final table | Cell 13 |
| Every row carries `run_id`, `computed_date`, `notebook_version`, `city_id`, `quality_status` | Cell 12 |
| No table exceeds `MAX_OUTPUT_ROWS` (500,000) | Cell 12 |
| Non-PRODUCTION modes cannot write to the production prefix | Cell 1 |

**What downstream guarantees back**

1. **The loader refreshes, it does not append.** Per table:
   `SAVEPOINT` -> `DELETE FROM <t> WHERE city_id=:c` -> chunked `INSERT`, with a
   single `COMMIT` across all tables. Readers see the previous snapshot until
   that commit, all tables flip together, and a table that errors rolls back to
   its savepoint and keeps yesterday's rows rather than going empty.

2. **The loader is triggered by `run_complete.json`, not by `manifest.json`.**
   A manifest trigger fires once per table - 20 invocations per run, the early
   ones racing tables that are still uploading.

4. **Nothing downstream may reference the `id` column.** It is `BIGSERIAL` and
   is regenerated by every daily replace. Dashboard links, ServiceNow payloads
   and saved views key on the declared primary key, never on `id`.

5. **S3 is the system of record; RDS is a serving cache.** RDS holds only the
   latest run. Historical snapshots are recoverable from S3 alone. Do not add
   append-mode to the loader to preserve history - that is the wrong layer.

6. **`/ps2/status` reports load coherence.** It reads `DISTINCT run_id,
   computed_date, notebook_version` per table, using the columns Cell 12
   already writes. A partial load shows as more than one `run_id`.

7. **The PS2, PS4 and PS5 loaders are independent.** Separate Lambdas,
   separate S3 prefixes, separate table sets. They share only the RDS
   connection pool, so the only cross-effect to watch is concurrent connection
   count when their schedules overlap.

**Operating the schedule**

- `PS2_RUN_MODE=PRODUCTION` is supplied by the job, never hardcoded here.
- Leave `PS2_MAX_SOURCE_LAG_DAYS` at 3. The freshness warning is expected while
  the client dump is fixed at 2026-04-11; widening the threshold would disarm
  the alarm for the day a refreshed feed goes stale.